In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde
from sklearn.mixture import GaussianMixture
import warnings
import copy
warnings.filterwarnings('ignore')


class SubsetOptimizer:
    """
    串行子集优化器：支持变大小重组（删N加M，最终锁定320-340）
    """
    
    def __init__(self,
                 column_select_mode='fc_only',
                 stat_threshold=1.0,
                 judge_tolerance=0.015,
                 size_min=320,
                 size_max=340,
                 max_iterations=2000,
                 initial_temperature=0.5,
                 random_seed=42):
        
        self.stat_threshold = stat_threshold
        self.judge_tolerance = judge_tolerance
        self.size_min = size_min
        self.size_max = size_max
        self.max_iterations = max_iterations
        self.initial_temperature = initial_temperature
        self.column_select_mode = column_select_mode
        
        np.random.seed(random_seed)
        
        self.full_df = None
        self.columns = None
        self.target_dict = None
    
    # ==================== 列筛选 ====================
    
    def select_columns(self, data_ref):
        all_cols = data_ref.columns[data_ref.columns.get_loc('EFL'):]
        if self.column_select_mode == 'fc_only':
            fc_cols = [c for c in all_cols if str(c).startswith('FC')][:93]
            if fc_cols:
                print(f"[列筛选] FC模式，共{len(fc_cols)}列")
                return fc_cols
            print("[列筛选] 无FC列，回退到first_93")
        return list(all_cols[:93])
    
    # ==================== 数值保护 ====================
    
    @staticmethod
    def safe_concat(base_df, add_df, numeric_cols):
        merged = pd.concat([base_df, add_df]).reset_index(drop=True)
        for c in numeric_cols:
            if c in merged.columns:
                merged[c] = pd.to_numeric(merged[c], errors='coerce')
        return merged
    
    @staticmethod
    def ensure_numeric(df, columns):
        """强制将指定列转为数值类型，无法转换的设为NaN，返回副本"""
        df = df.copy()
        for col in columns:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        return df
    
    # ==================== 双峰检测 ====================
    
    @staticmethod
    def is_bimodal(series):
        data = series.dropna().values.flatten()
        if len(data) < 20:
            return False
        try:
            std = data.std(ddof=1)
            if not std or std == 0:
                return False
            bw = max(0.5 * std, (data.max() - data.min()) / 50)
            kde = gaussian_kde(data, bw_method=bw / std)
            xs = np.linspace(data.min(), data.max(), 2000)
            vals = kde(xs)
            
            peaks = [(i, vals[i]) for i in range(1, len(vals)-1)
                     if vals[i] > vals[i-1] and vals[i] > vals[i+1]
                     and vals[i] > vals.max() * 0.1]
            if len(peaks) <= 1:
                return False
            
            peaks.sort(key=lambda x: x[1], reverse=True)
            dist = abs(xs[peaks[0][0]] - xs[peaks[1][0]])
            return dist / (data.max() - data.min()) > 0.05
        except:
            return False
    
    # ==================== 统计量计算 ====================
    
    def calc_stats(self, df):
        stats = {}
        for col in self.columns:
            if col not in df.columns:
                continue
            s = pd.to_numeric(df[col], errors='coerce')
            if s.notna().sum() == 0:
                continue
            stats[col] = {
                'median': s.median(),
                'q1': s.quantile(0.25),
                'q3': s.quantile(0.75),
                'std': s.std(ddof=1) if s.notna().sum() > 1 else 0.0,
            }
        return stats
    
    # ==================== 约束评估 ====================
    
    def evaluate(self, df):
        n = len(df)
        size_ok = self.size_min <= n <= self.size_max
        
        vc = df['Judge'].value_counts()
        best_p = vc.get('BEST', 0) / n if n else 0
        pass_p = vc.get('PASS', 0) / n if n else 0
        medium_p = vc.get('MEDIUM', 0) / n if n else 0
        
        judge_ok = ((0.45 - self.judge_tolerance) <= best_p <= (0.50 + self.judge_tolerance) and
                    (0.42 - self.judge_tolerance) <= pass_p <= (0.47 + self.judge_tolerance) and
                    abs(medium_p - 0.08) <= self.judge_tolerance)
        
        stats = self.calc_stats(df)
        violations = []
        for col in self.columns:
            if col not in stats or col not in self.target_dict:
                continue
            d = max(abs(stats[col]['median'] - self.target_dict[col]['median']),
                    abs(stats[col]['q1'] - self.target_dict[col]['q1']),
                    abs(stats[col]['q3'] - self.target_dict[col]['q3']))
            if d > self.stat_threshold:
                violations.append({'col': col, 'diff': d})
        stat_ok = len(violations) == 0
        
        bimodal_cols = [c for c in self.columns if c in df.columns and self.is_bimodal(df[c])]
        dist_ok = len(bimodal_cols) == 0
        
        median_diffs = [abs(stats[c]['median'] - self.target_dict[c]['median'])
                        for c in self.columns if c in stats and c in self.target_dict]
        std_diffs = [abs(stats[c]['std'] - self.target_dict[c]['std'])
                     for c in self.columns if c in stats and c in self.target_dict]
        
        avg_median = np.mean(median_diffs) if median_diffs else float('inf')
        avg_std = np.mean(std_diffs) if std_diffs else float('inf')
        
        return {
            'size_ok': size_ok, 'judge_ok': judge_ok, 'stat_ok': stat_ok, 'dist_ok': dist_ok,
            'all_ok': size_ok and judge_ok and stat_ok and dist_ok,
            'violations': violations, 'bimodal_cols': bimodal_cols,
            'size': n, 'avg_median_diff': avg_median, 'avg_std_diff': avg_std,
            'judge_score': (best_p-0.475)**2 + (pass_p-0.445)**2 + (medium_p-0.08)**2,
            'stats': stats,
        }
    
    # ==================== 分层采样 ====================
    
    def stratified_sample(self, df, target_size):
        groups = {j: df[df['Judge'] == j] for j in ['BEST', 'PASS', 'MEDIUM']}
        best_n = min(int(target_size * 0.475), len(groups['BEST']))
        pass_n = min(int(target_size * 0.445), len(groups['PASS']))
        medium_n = min(target_size - best_n - pass_n, len(groups['MEDIUM']))
        
        short = target_size - (best_n + pass_n + medium_n)
        if short > 0:
            extra_best = min(short // 2, len(groups['BEST']) - best_n)
            best_n += extra_best
            short -= extra_best
            extra_pass = min(short, len(groups['PASS']) - pass_n)
            pass_n += extra_pass
        
        parts = []
        if best_n > 0: parts.append(groups['BEST'].sample(n=best_n))
        if pass_n > 0: parts.append(groups['PASS'].sample(n=pass_n))
        if medium_n > 0: parts.append(groups['MEDIUM'].sample(n=medium_n))
        
        if not parts:
            return df.iloc[:0]
        result = pd.concat(parts).sample(frac=1).reset_index(drop=True)
        return self._adjust_size(result, df)
    
    def _adjust_size(self, subset, full):
        """将子集大小调整到范围内，优先保持Judge比例"""
        # 先检查是否需要补充
        if len(subset) < self.size_min:
            need = self.size_min - len(subset)
            avail = full[~full.index.isin(subset.index)]
            if len(avail) >= need:
                # 优先补充缺的Judge类型
                subset = self._replenish_by_judge(subset, avail, need)
        
        # 再检查是否需要删除
        elif len(subset) > self.size_max:
            excess = len(subset) - self.size_max
            # 随机删除，但尽量保留Judge比例
            subset = self._trim_by_judge(subset, excess)
        
        return subset.reset_index(drop=True)
    
    def _replenish_by_judge(self, subset, avail, need):
        """按Judge比例补充样本"""
        best_p, pass_p, medium_p, _ = self._get_judge_props(subset)
        result = [subset]
        remaining = need
        
        # 优先补充比例偏低的类型
        deficits = []
        if best_p < 0.45:
            deficits.append(('BEST', 0.475 - best_p))
        if pass_p < 0.42:
            deficits.append(('PASS', 0.445 - pass_p))
        if medium_p < 0.065:
            deficits.append(('MEDIUM', 0.08 - medium_p))
        
        # 按缺口从大到小排序
        deficits.sort(key=lambda x: x[1], reverse=True)
        
        for judge_type, _ in deficits:
            if remaining <= 0:
                break
            pool = avail[avail['Judge'] == judge_type]
            if len(pool) > 0:
                n_take = min(remaining, len(pool))
                result.append(pool.sample(n=n_take))
                remaining -= n_take
        
        # 如果还有缺口，随机补充
        if remaining > 0:
            used_idx = pd.concat(result).index
            left = avail[~avail.index.isin(used_idx)]
            if len(left) >= remaining:
                result.append(left.sample(n=remaining))
        
        return pd.concat(result).reset_index(drop=True)
    
    def _trim_by_judge(self, subset, excess):
        """按Judge比例删除样本，优先删除比例偏高的类型"""
        best_p, pass_p, medium_p, _ = self._get_judge_props(subset)
        
        # 计算各类型超出量
        surpluses = []
        if best_p > 0.50:
            surpluses.append(('BEST', best_p - 0.50))
        if pass_p > 0.47:
            surpluses.append(('PASS', pass_p - 0.47))
        if medium_p > 0.095:
            surpluses.append(('MEDIUM', medium_p - 0.08 - self.judge_tolerance))
        
        to_drop = []
        surpluses.sort(key=lambda x: x[1], reverse=True)
        
        for judge_type, _ in surpluses:
            if len(to_drop) >= excess:
                break
            pool = subset[subset['Judge'] == judge_type]
            n_drop = min(excess - len(to_drop), len(pool))
            to_drop.extend(pool.sample(n=n_drop).index.tolist())
        
        # 如果还不够，随机删除
        if len(to_drop) < excess:
            remaining = subset.drop(to_drop)
            n_more = excess - len(to_drop)
            to_drop.extend(remaining.sample(n=n_more).index.tolist())
        
        return subset.drop(to_drop)
    
    # ==================== 瓶颈列与双峰处理 ====================
    
    def _get_bottleneck(self, stats):
        diffs = []
        for col in self.columns:
            if col not in stats or col not in self.target_dict:
                continue
            d = max(abs(stats[col]['median'] - self.target_dict[col]['median']),
                    abs(stats[col]['q1'] - self.target_dict[col]['q1']),
                    abs(stats[col]['q3'] - self.target_dict[col]['q3']))
            if d > self.stat_threshold:
                diffs.append((col, d, stats[col]['median'] - self.target_dict[col]['median']))
        return max(diffs, key=lambda x: x[1]) if diffs else None
    
    def _deflash_bimodal(self, df, col):
        s = pd.to_numeric(df[col], errors='coerce')
        data = s.dropna().values.reshape(-1, 1)
        if len(data) < 20:
            return df, False
        try:
            labels = GaussianMixture(n_components=2, random_state=42).fit_predict(data)
        except:
            return df, False
        
        mask = s.notna()
        labeled = df[mask].copy()
        labeled['_label'] = labels
        
        g0, g1 = labeled[labeled['_label']==0], labeled[labeled['_label']==1]
        if len(g0) == 0 or len(g1) == 0:
            return df, False
        
        small = 0 if len(g0) < len(g1) else 1
        to_remove = labeled[labeled['_label'] == small]
        target_m = self.target_dict[col]['median']
        to_remove['_dist'] = abs(pd.to_numeric(to_remove[col], errors='coerce') - target_m)
        to_remove = to_remove.sort_values('_dist', ascending=False)
        
        new_df = df.copy()
        for idx in to_remove.index:
            if len(new_df) <= self.size_min:
                break
            temp = new_df.drop(idx)
            if not self._check_judge_quick(temp):
                continue
            new_df = temp
            if not self.is_bimodal(new_df[col]):
                break
        
        new_df = new_df.drop(columns=[c for c in ['_label', '_dist'] if c in new_df.columns], errors='ignore')
        return self._adjust_size(new_df, self.full_df), not self.is_bimodal(new_df[col])
    
    def _check_judge_quick(self, df):
        n = len(df)
        if n == 0:
            return False
        vc = df['Judge'].value_counts()
        bp = vc.get('BEST', 0) / n
        pp = vc.get('PASS', 0) / n
        mp = vc.get('MEDIUM', 0) / n
        return ((0.45-self.judge_tolerance) <= bp <= (0.50+self.judge_tolerance) and
                (0.42-self.judge_tolerance) <= pp <= (0.47+self.judge_tolerance) and
                abs(mp-0.08) <= self.judge_tolerance)
    
    def _accept(self, new_ev, cur_ev, temp):
        if new_ev['all_ok'] and not cur_ev['all_ok']:
            return True
        if new_ev['all_ok'] and cur_ev['all_ok']:
            return (new_ev['avg_median_diff'] < cur_ev['avg_median_diff'] - 1e-6 or
                    (abs(new_ev['avg_median_diff'] - cur_ev['avg_median_diff']) < 1e-6 and
                     new_ev['avg_std_diff'] < cur_ev['avg_std_diff'] - 1e-6))
        if not new_ev['all_ok'] and not cur_ev['all_ok']:
            new_v = len(new_ev['violations']) + (0 if new_ev['judge_ok'] else 1) + len(new_ev['bimodal_cols'])
            cur_v = len(cur_ev['violations']) + (0 if cur_ev['judge_ok'] else 1) + len(cur_ev['bimodal_cols'])
            if new_v < cur_v:
                return True
            if new_v == cur_v and temp > 0.01 and np.random.random() < np.exp(-0.1/temp):
                return True
        return False
    
    # ==================== 核心：变大小贪心优化 ====================
    
    def optimize_with_early_stop(self, init_df, early_stop_rounds=200):
        """
        变大小重组优化：
        - 可以批量删除N个样本
        - 可以批量添加M个样本
        - 可以只删不加、只加不删、或删N加M
        - 每次操作后用 _adjust_size 确保最终大小在范围内
        """
        current = init_df.copy()
        cur_eval = self.evaluate(current)
        
        best_local = current.copy()
        best_local_eval = copy.deepcopy(cur_eval)
        
        no_improve = 0
        
        for it in range(self.max_iterations):
            temp = self.initial_temperature * (1 - it / self.max_iterations)
            improved = False
            
            # ===== 策略1: 修复双峰（批量删除小峰） =====
            if not cur_eval['dist_ok'] and cur_eval['bimodal_cols']:
                for bcol in cur_eval['bimodal_cols'][:2]:
                    if len(current) <= self.size_min:
                        break
                    new_df, ok = self._deflash_bimodal(current, bcol)
                    # _deflash_bimodal 内部已经调用了 _adjust_size
                    new_eval = self.evaluate(new_df)
                    if new_eval['size_ok'] and new_eval['judge_score'] <= cur_eval['judge_score'] + 0.01:
                        current, cur_eval = new_df, new_eval
                        if ok:
                            improved = True
                            break
            
            # ===== 策略2: 修复统计量（批量删除极端值 + 批量补充反向值） =====
            if not improved and not cur_eval['stat_ok']:
                bn = self._get_bottleneck(cur_eval['stats'])
                if bn:
                    col, diff, direction = bn
                    
                    # 动态决定删除数量：根据偏差大小决定
                    # 偏差大就多删，偏差小就少删
                    n_remove = min(np.random.randint(1, 6), len(current) - self.size_min)
                    # 也可以根据 diff 决定：n_remove = min(int(diff), 5, len(current) - self.size_min)
                    
                    try:
                        if direction > 0:  # median偏高，删大值
                            drop_pool = current.nlargest(n_remove, col)
                        else:  # median偏低，删小值
                            drop_pool = current.nsmallest(n_remove, col)
                    except:
                        drop_pool = current.sort_values(col, ascending=(direction <= 0)).head(n_remove)
                    
                    # 删除
                    temp_df = current.drop(drop_pool.index)
                    
                    # 动态决定添加数量：不一定和删除数量相等
                    # 可以添0个（只收缩）、添相同数量（保持大小）、或添更多（扩张）
                    avail = self.full_df[~self.full_df.index.isin(temp_df.index)]
                    if len(avail) == 0:
                        # 没有可添加的，只做删除，然后调整大小
                        new_df = self._adjust_size(temp_df, self.full_df)
                        new_eval = self.evaluate(new_df)
                        if self._accept(new_eval, cur_eval, temp):
                            current, cur_eval = new_df, new_eval
                            improved = True
                    else:
                        # 决定添加数量：随机1-5个，或者让最终大小落在范围内
                        max_add = min(5, len(avail), self.size_max - len(temp_df))
                        if max_add > 0:
                            n_add = np.random.randint(1, max_add + 1)
                        else:
                            n_add = 0
                        
                        if n_add > 0:
                            avail = ensure_numeric_static(avail, [col])
                            try:
                                if direction > 0:  # median偏高，加小值
                                    add_pool = avail.nsmallest(min(20, len(avail)), col)
                                else:  # median偏低，加大值
                                    add_pool = avail.nlargest(min(20, len(avail)), col)
                            except:
                                add_pool = avail.sort_values(col, ascending=(direction > 0)).head(20)
                            
                            # Judge比例修复
                            bp, pp, mp, _ = self._get_judge_props(temp_df)
                            need = None
                            if bp < 0.45: need = 'BEST'
                            elif pp < 0.42: need = 'PASS'
                            elif mp < 0.065: need = 'MEDIUM'
                            if need:
                                pref = add_pool[add_pool['Judge'] == need]
                                if len(pref) > 0:
                                    add_pool = pref
                            
                            # 从候选池中选n_add个
                            if len(add_pool) >= n_add:
                                selected = add_pool.sample(n=n_add)
                                new_df = self.safe_concat(temp_df, selected, self.columns)
                            else:
                                new_df = self.safe_concat(temp_df, add_pool, self.columns)
                        else:
                            new_df = temp_df.copy()
                        
                        # 确保大小在范围内（如果删多了或加多了）
                        new_df = self._adjust_size(new_df, self.full_df)
                        new_eval = self.evaluate(new_df)
                        
                        if self._accept(new_eval, cur_eval, temp):
                            current, cur_eval = new_df, new_eval
                            improved = True
            
            # ===== 策略3: 全部满足后精调（批量随机替换） =====
            if not improved and cur_eval['all_ok']:
                # 尝试批量替换：删k个，加m个
                for _ in range(3):
                    # 随机决定删几个（1-5）
                    k = min(np.random.randint(1, 6), len(current) - self.size_min)
                    drop_idx = current.sample(n=k).index
                    temp_df = current.drop(drop_idx)
                    
                    avail = self.full_df[~self.full_df.index.isin(temp_df.index)]
                    if len(avail) == 0:
                        break
                    
                    # 随机决定加几个（0到min(5, avail)）
                    max_add = min(5, len(avail), self.size_max - len(temp_df))
                    m = np.random.randint(0, max_add + 1) if max_add > 0 else 0
                    
                    if m > 0:
                        try_pool = avail.sample(min(20, len(avail)))
                        # 优先选能让目标函数改善的
                        best_new = None
                        best_new_eval = None
                        for _, row in try_pool.iterrows():
                            test_df = self.safe_concat(temp_df, row.to_frame().T, self.columns)
                            if not self.size_min <= len(test_df) <= self.size_max:
                                continue
                            test_eval = self.evaluate(test_df)
                            if test_eval['all_ok']:
                                if (best_new_eval is None or 
                                    test_eval['avg_median_diff'] < best_new_eval['avg_median_diff'] - 1e-6 or
                                    (abs(test_eval['avg_median_diff'] - best_new_eval['avg_median_diff']) < 1e-6 and
                                     test_eval['avg_std_diff'] < best_new_eval['avg_std_diff'] - 1e-6)):
                                    best_new = test_df
                                    best_new_eval = test_eval
                        
                        if best_new is not None:
                            # 继续尝试加更多，看能否更好
                            new_df = best_new.copy()
                            for _ in range(m - 1):
                                remaining_avail = self.full_df[~self.full_df.index.isin(new_df.index)]
                                if len(remaining_avail) == 0:
                                    break
                                try_add = remaining_avail.sample(min(10, len(remaining_avail)))
                                found = False
                                for _, row in try_add.iterrows():
                                    test_df = self.safe_concat(new_df, row.to_frame().T, self.columns)
                                    if not self.size_min <= len(test_df) <= self.size_max:
                                        continue
                                    test_eval = self.evaluate(test_df)
                                    if test_eval['all_ok'] and test_eval['avg_median_diff'] < self.evaluate(new_df)['avg_median_diff'] - 1e-6:
                                        new_df = test_df
                                        found = True
                                        break
                                if not found:
                                    break
                            
                            new_eval = self.evaluate(new_df)
                            if new_eval['all_ok'] and (new_eval['avg_median_diff'] < cur_eval['avg_median_diff'] - 1e-6 or
                                (abs(new_eval['avg_median_diff'] - cur_eval['avg_median_diff']) < 1e-6 and
                                 new_eval['avg_std_diff'] < cur_eval['avg_std_diff'] - 1e-6)):
                                current, cur_eval = new_df, new_eval
                                improved = True
                                break
                    else:
                        # 只删不加（收缩）
                        new_df = self._adjust_size(temp_df, self.full_df)
                        new_eval = self.evaluate(new_df)
                        if new_eval['all_ok'] and (new_eval['avg_median_diff'] < cur_eval['avg_median_diff'] - 1e-6 or
                            (abs(new_eval['avg_median_diff'] - cur_eval['avg_median_diff']) < 1e-6 and
                             new_eval['avg_std_diff'] < cur_eval['avg_std_diff'] - 1e-6)):
                            current, cur_eval = new_df, new_eval
                            improved = True
                            break
            
            # 更新局部最优
            if cur_eval['avg_median_diff'] != float('inf'):
                if cur_eval['all_ok'] and (not best_local_eval['all_ok'] or 
                    cur_eval['avg_median_diff'] < best_local_eval['avg_median_diff'] - 1e-6 or
                    (abs(cur_eval['avg_median_diff'] - best_local_eval['avg_median_diff']) < 1e-6 and
                     cur_eval['avg_std_diff'] < best_local_eval['avg_std_diff'] - 1e-6)):
                    best_local, best_local_eval = current.copy(), copy.deepcopy(cur_eval)
            
            if improved:
                no_improve = 0
            else:
                no_improve += 1
            
            # 早停
            if no_improve >= early_stop_rounds:
                print(f"  [早停] 连续{early_stop_rounds}轮无改进，第{it+1}轮退出")
                break
            
            # 大扰动：批量换血
            if no_improve >= 80 and it < self.max_iterations - 100:
                n_drop = np.random.randint(3, max(4, len(current) // 10))  # 批量删3-10%
                drop_idx = current.sample(n=min(n_drop, len(current) - self.size_min)).index
                temp_df = current.drop(drop_idx)
                
                avail = self.full_df[~self.full_df.index.isin(temp_df.index)]
                if len(avail) > 0:
                    n_add = np.random.randint(0, min(len(avail), self.size_max - len(temp_df)) + 1)
                    if n_add > 0:
                        current = self.safe_concat(temp_df, avail.sample(n=n_add), self.columns)
                    else:
                        current = temp_df.copy()
                    current = self._adjust_size(current, self.full_df)
                    cur_eval = self.evaluate(current)
                    no_improve = 0
        
        # fallback
        if best_local_eval['avg_median_diff'] == float('inf') and cur_eval['avg_median_diff'] != float('inf'):
            best_local, best_local_eval = current.copy(), copy.deepcopy(cur_eval)
        
        return best_local, best_local_eval
    
    @staticmethod
    def _get_judge_props(df):
        n = len(df)
        if n == 0:
            return 0, 0, 0, 0
        vc = df['Judge'].value_counts()
        return (vc.get('BEST', 0)/n, vc.get('PASS', 0)/n, vc.get('MEDIUM', 0)/n,
                n - vc.get('BEST', 0) - vc.get('PASS', 0) - vc.get('MEDIUM', 0))
    
    # ==================== 串行主搜索 ====================
    
    def search_serial(self, df, target_dict, columns_of_interest, n_runs=10, early_stop=200):
        self.full_df = df
        self.columns = list(columns_of_interest)
        self.target_dict = {k: v for k, v in target_dict.items() if k in self.columns}
        
        n_total = len(df)
        print(f"\n{'='*60}")
        print(f"串行搜索开始 | 样本{n_total} | 列{len(self.columns)} | 共{n_runs}轮")
        print(f"变大小重组模式: 可批量删N加M，最终锁定{self.size_min}-{self.size_max}")
        print(f"每轮早停阈值: 连续{early_stop}轮无改进则退出")
        print(f"{'='*60}")
        
        print(f"\n原始数据Judge分布:")
        for j, c in df['Judge'].value_counts().items():
            print(f"  {j}: {c} ({c/n_total*100:.1f}%)")
        
        others = set(df['Judge'].unique()) - {'BEST', 'PASS', 'MEDIUM'}
        if others:
            print(f"  警告: 其他Judge取值: {others}")
        
        limits = [len(df[df['Judge']==j]) / r for j, r in [('MEDIUM',0.08),('BEST',0.45),('PASS',0.42)]]
        theoretical_max = int(min(limits))
        print(f"\n理论最大子集大小: {theoretical_max}")
        if theoretical_max < self.size_min:
            print("错误: Judge数量不足以构建目标大小的子集")
            return None
        
        best_result = None
        best_eval = None
        history = []
        
        size_pool = [s for s in range(self.size_min, min(self.size_max, theoretical_max)+1, 5)]
        if not size_pool:
            size_pool = [self.size_min]
        
        for run in range(n_runs):
            init_size = size_pool[run % len(size_pool)]
            
            print(f"\n{'='*50}")
            print(f"第 {run+1}/{n_runs} 轮 | 初始大小: {init_size}")
            print(f"{'='*50}")
            
            try:
                init_df = self.stratified_sample(df, init_size)
                init_eval = self.evaluate(init_df)
                print(f"[初始化] size={len(init_df)}, all_ok={init_eval['all_ok']}, "
                      f"median_diff={init_eval['avg_median_diff']:.4f}, "
                      f"violations={len(init_eval['violations'])}, "
                      f"bimodal={len(init_eval['bimodal_cols'])}")
                
                result_df, result_eval = self.optimize_with_early_stop(init_df, early_stop_rounds=early_stop)
                
                print(f"[优化结束] size={len(result_df)}, all_ok={result_eval['all_ok']}, "
                      f"median_diff={result_eval['avg_median_diff']:.4f}, "
                      f"std_diff={result_eval['avg_std_diff']:.4f}")
                
                history.append({
                    'run': run + 1,
                    'init_size': init_size,
                    'final_size': len(result_df),
                    'all_ok': result_eval['all_ok'],
                    'median_diff': result_eval['avg_median_diff'],
                    'std_diff': result_eval['avg_std_diff'],
                    'violations': len(result_eval['violations']),
                    'bimodal_cols': len(result_eval['bimodal_cols']),
                })
                
                if result_eval['all_ok']:
                    is_better = (
                        best_eval is None or
                        result_eval['avg_median_diff'] < best_eval['avg_median_diff'] - 1e-6 or
                        (abs(result_eval['avg_median_diff'] - best_eval['avg_median_diff']) < 1e-6 and
                         result_eval['avg_std_diff'] < best_eval['avg_std_diff'] - 1e-6)
                    )
                    if is_better:
                        best_result = result_df.copy()
                        best_eval = copy.deepcopy(result_eval)
                        print(f"★ [更新最优] 第{run+1}轮 → median_diff={result_eval['avg_median_diff']:.6f}, "
                              f"std_diff={result_eval['avg_std_diff']:.6f}")
                
                if best_eval and best_eval['avg_median_diff'] < 0.001:
                    print(f"\n[全局早停] 第{run+1}轮已找到足够优的解，提前终止")
                    break
                    
            except Exception as e:
                print(f"[第{run+1}轮] 异常: {e}")
                history.append({
                    'run': run + 1, 'init_size': init_size, 'final_size': 0,
                    'all_ok': False, 'median_diff': float('inf'),
                    'std_diff': float('inf'), 'violations': 0, 'bimodal_cols': 0,
                })
        
        self._print_history(history)
        
        if best_result is not None:
            self._report_final(best_result, best_eval)
            return best_result
        else:
            print(f"\n{'='*60}")
            print("警告: 所有轮次均未找到满足全部约束的子集")
            print(f"{'='*60}")
            valid_hist = [h for h in history if h['median_diff'] != float('inf')]
            if valid_hist:
                best_hist = min(valid_hist, key=lambda x: (0 if x['all_ok'] else 1, x['median_diff'], x['std_diff']))
                print(f"历史最优（未完全满足）: 第{best_hist['run']}轮")
            return None
    
    def _print_history(self, history):
        print(f"\n{'='*70}")
        print("搜索历史汇总")
        print(f"{'='*70}")
        print(f"{'轮次':>4} {'初始大小':>8} {'最终大小':>8} {'满足约束':>8} "
              f"{'median差':>10} {'std差':>10} {'违规数':>6} {'双峰数':>6}")
        print("-" * 70)
        for h in history:
            ok_mark = "✓" if h['all_ok'] else "✗"
            print(f"{h['run']:>4} {h['init_size']:>8} {h['final_size']:>8} {ok_mark:>8} "
                  f"{h['median_diff']:>10.4f} {h['std_diff']:>10.4f} "
                  f"{h['violations']:>6} {h['bimodal_cols']:>6}")
        print("-" * 70)
        ok_runs = [h for h in history if h['all_ok']]
        print(f"满足约束的轮次: {len(ok_runs)}/{len(history)}")
        if ok_runs:
            best_hist = min(ok_runs, key=lambda x: (x['median_diff'], x['std_diff']))
            print(f"历史最优轮次: 第{best_hist['run']}轮 (median_diff={best_hist['median_diff']:.4f})")
    
    def _report_final(self, df, ev):
        print(f"\n{'='*60}")
        print("最终最优子集报告")
        print(f"{'='*60}")
        print(f"样本数: {len(df)}")
        print(f"约束: 大小{'✓' if ev['size_ok'] else '✗'} | "
              f"Judge{'✓' if ev['judge_ok'] else '✗'} | "
              f"统计量{'✓' if ev['stat_ok'] else '✗'}({len(ev['violations'])}未通过) | "
              f"双峰{'✓' if ev['dist_ok'] else '✗'}({len(ev['bimodal_cols'])}列)")
        print(f"全部满足: {'✓' if ev['all_ok'] else '✗'}")
        
        n = len(df)
        vc = df['Judge'].value_counts()
        print(f"\nJudge比例: BEST={vc.get('BEST',0)/n:.3f} PASS={vc.get('PASS',0)/n:.3f} "
              f"MEDIUM={vc.get('MEDIUM',0)/n:.3f}")
        print(f"目标函数: 平均median差={ev['avg_median_diff']:.6f} | 平均std差={ev['avg_std_diff']:.6f}")
        
        if ev['violations']:
            print(f"\n未通过列(Top5):")
            for v in sorted(ev['violations'], key=lambda x: x['diff'], reverse=True)[:5]:
                print(f"  {v['col']}: max_diff={v['diff']:.4f}")


def ensure_numeric_static(df, columns):
    """静态版本，用于类外部调用"""
    df = df.copy()
    for c in columns:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

# ==================== 主程序 ====================

if __name__ == "__main__":
    
    df_C = pd.read_excel(
        r'C:\Users\HP\Desktop\code\C.csv',
        engine='openpyxl'
    )
    df_C30 = pd.read_excel(
        r'C:\Users\HP\Desktop\code\C30.csv',
        engine='openpyxl'
    )
    
    data_ref = df_C30[df_C30['#'] == '0.0XM C3.0_CGS']
    
    optimizer = SubsetOptimizer(
        column_select_mode='fc_only',
        stat_threshold=3.0,
        judge_tolerance=0.015,
        size_min=320,
        size_max=340,
        max_iterations=50,
        initial_temperature=0.5,
        random_seed=42,
    )
    
    columns_of_interest = optimizer.select_columns(data_ref)
    
    target_stats = {}
    for col in columns_of_interest:
        if data_ref[col].dtype in ['float64', 'int64']:
            target_stats[col] = {
                'median': data_ref[col].median(),
                'q3': data_ref[col].quantile(0.75),
                'q1': data_ref[col].quantile(0.25),
                'mean': data_ref[col].mean(),
                'std': data_ref[col].std(ddof=1) if data_ref[col].count() > 1 else 0.0,
            }
    
    counts = df_C['#'].value_counts()
    large = counts[counts > 300].index.tolist()
    if not large:
        print("未找到样本量大于300的组")
        exit()
    
    sample_id = large[0]
    print(f"选择样本组: {sample_id} (n={counts[sample_id]})")
    df_subset = df_C[df_C['#'] == sample_id].copy()
    
    result = optimizer.search_serial(
        df_subset, target_stats, columns_of_interest,
        n_runs=5,
        early_stop=30
    )
    
    if result is not None:
        out = r'C:\Users\HP\Desktop\code\optimal_subset5.xlsx'
        result.to_excel(out, index=False)
        print(f"\n结果已保存: {out}")
        
        final_stats = optimizer.calc_stats(result)
        diffs = []
        for col in columns_of_interest:
            if col in final_stats and col in target_stats:
                diffs.append({
                    'column': col,
                    'median_diff': abs(final_stats[col]['median'] - target_stats[col]['median']),
                    'q1_diff': abs(final_stats[col]['q1'] - target_stats[col]['q1']),
                    'q3_diff': abs(final_stats[col]['q3'] - target_stats[col]['q3']),
                    'std_diff': abs(final_stats[col]['std'] - target_stats[col]['std']),
                })
        pd.DataFrame(diffs).to_excel(
            r'C:\Users\HP\Desktop\code\diff_analysis5.xlsx', index=False
        )

In [19]:
import math
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew

warnings.filterwarnings("ignore", category=RuntimeWarning)


def load_table(path: str) -> pd.DataFrame:
    p = str(path).lower()
    if p.endswith(".csv"):
        return pd.read_csv(path, low_memory=False)
    return pd.read_excel(path, engine="openpyxl")


class FastSubsetOptimizer:
    """
    从 df_C 中抽取 320~340（默认约330）样本，使其分布和统计量尽量接近 df_ref。
    约束：
    - Judge 比例强制满足
    - 所有 93 列不能双峰
    - FC 列尽量接近正态
    - 对参考双峰列：不要求形态相似，只要求统计量相近
    """

    def __init__(
        self,
        target_mode: str = "all_93",      # "all_93" or "fc_only"
        size_min: int = 320,
        size_max: int = 340,
        target_size: int = 330,
        judge_tolerance: float = 0.015,
        stat_threshold: float = 3.0,
        n_starts: int = 5,
        max_iter: int = 800,
        early_stop: int = 180,
        init_temp: float = 0.08,
        cooling: float = 0.997,
        normal_skew_thr: float = 0.5,
        normal_kurt_thr: float = 1.5,
        random_seed: int = 42,
        w_shape: float = 1.2,
        w_stat: float = 1.0,
        w_norm: float = 0.6,
        w_stat_violation: float = 0.2,
    ):
        self.target_mode = target_mode
        self.size_min = size_min
        self.size_max = size_max
        self.target_size = target_size
        self.judge_tolerance = judge_tolerance
        self.stat_threshold = stat_threshold

        self.n_starts = n_starts
        self.max_iter = max_iter
        self.early_stop = early_stop
        self.init_temp = init_temp
        self.cooling = cooling

        self.normal_skew_thr = normal_skew_thr
        self.normal_kurt_thr = normal_kurt_thr

        self.w_shape = w_shape
        self.w_stat = w_stat
        self.w_norm = w_norm
        self.w_stat_violation = w_stat_violation

        self.rng = np.random.default_rng(random_seed)

        # Populated in _prepare
        self.df_c = None
        self.df_ref = None
        self.all_cols = None
        self.fc_cols = None
        self.target_cols = None

        self.X_all = None
        self.X_ref_all = None
        self.judge_code = None
        self.pool = None
        self.n_rows = 0

        self.all_pos = None
        self.target_pos = None
        self.fc_pos_all = None

        # Target stats (for optimization columns)
        self.t_q1 = None
        self.t_med = None
        self.t_q3 = None
        self.t_std = None
        self.t_shape = None
        self.t_scale = None
        self.t_bimodal = None

        # All-93 reference stats (for final report)
        self.ref_q1_all = None
        self.ref_med_all = None
        self.ref_q3_all = None
        self.ref_std_all = None
        self.ref_shape_all = None
        self.scale_all = None
        self.ref_bimodal_all = None

        # Fixed composition for fast swap search
        self.fixed_n = None
        self.fixed_counts = None  # dict {0: best_n, 1: pass_n, 2: medium_n}

        self.shape_q = np.array([5, 15, 25, 35, 50, 65, 75, 85, 95], dtype=np.float64)

    @staticmethod
    def _extract_93_columns(df: pd.DataFrame) -> List[str]:
        if "EFL" not in df.columns:
            raise ValueError("未找到列 EFL，无法定位目标 93 列。")
        start = df.columns.get_loc("EFL")
        cols = list(df.columns[start:start + 93])
        if len(cols) != 93:
            raise ValueError(f"从 EFL 开始不足 93 列，实际只有 {len(cols)} 列。")
        return cols

    @staticmethod
    def _normalize_judge(s: pd.Series) -> pd.Series:
        # 兼容 BSET -> BEST
        x = s.astype(str).str.strip().str.upper()
        x = x.replace({"BSET": "BEST"})
        return x

    @staticmethod
    def _compute_stats(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        # X shape: [n_samples, n_cols]
        q = np.nanpercentile(X, [25, 50, 75], axis=0)
        q1, med, q3 = q[0], q[1], q[2]
        std = np.nanstd(X, axis=0, ddof=1)
        std = np.nan_to_num(std, nan=0.0, posinf=0.0, neginf=0.0)
        return q1, med, q3, std

    @staticmethod
    def _is_bimodal(
        x: np.ndarray,
        min_n: int = 30,
        peak_height_ratio: float = 0.12,
        sep_ratio: float = 0.15,
    ) -> bool:
        """
        速度优先的双峰检测：
        - 直方图 + 平滑 + 双峰峰值检测
        """
        y = np.asarray(x, dtype=np.float64)
        y = y[np.isfinite(y)]
        n = y.size
        if n < min_n:
            return False

        y_min = float(np.min(y))
        y_max = float(np.max(y))
        if (not np.isfinite(y_min)) or (not np.isfinite(y_max)) or (y_max <= y_min):
            return False

        bins = int(np.clip(np.sqrt(n), 12, 48))
        hist, edges = np.histogram(y, bins=bins)
        if hist.max() <= 0:
            return False

        smooth = np.convolve(hist.astype(np.float64), np.array([1.0, 2.0, 1.0]), mode="same")
        thr = peak_height_ratio * smooth.max()

        left = smooth[:-2]
        mid = smooth[1:-1]
        right = smooth[2:]
        peak_idx = np.where((mid > left) & (mid >= right) & (mid >= thr))[0] + 1

        if peak_idx.size < 2:
            return False

        top2 = peak_idx[np.argsort(smooth[peak_idx])[-2:]]
        centers = 0.5 * (edges[:-1] + edges[1:])
        sep = abs(centers[top2[0]] - centers[top2[1]]) / (y_max - y_min + 1e-12)
        return bool(sep >= sep_ratio)

    def _prepare(self, df_C: pd.DataFrame, df_ref: pd.DataFrame) -> None:
        self.df_c = df_C.copy()
        self.df_ref = df_ref.copy()

        # 1) 93 列与 FC 列
        self.all_cols = self._extract_93_columns(self.df_ref)
        missing = [c for c in self.all_cols if c not in self.df_c.columns]
        if missing:
            raise ValueError(f"df_C 缺少目标列: {missing[:5]} ...")

        self.fc_cols = [c for c in self.all_cols if str(c).startswith("FC")]
        if self.target_mode == "all_93":
            self.target_cols = list(self.all_cols)
        elif self.target_mode == "fc_only":
            if not self.fc_cols:
                raise ValueError("target_mode='fc_only' 但未找到 FC 列。")
            self.target_cols = list(self.fc_cols)
        else:
            raise ValueError("target_mode 只能是 'all_93' 或 'fc_only'。")

        # 2) Judge 标准化并过滤
        if "Judge" not in self.df_c.columns:
            raise ValueError("df_C 缺少 Judge 列。")
        self.df_c["Judge"] = self._normalize_judge(self.df_c["Judge"])
        valid_mask = self.df_c["Judge"].isin(["BEST", "PASS", "MEDIUM"])
        self.df_c = self.df_c.loc[valid_mask].copy()
        if self.df_c.empty:
            raise ValueError("df_C 在过滤 Judge 后为空。")

        # 保存原始索引
        self.df_c = self.df_c.reset_index(drop=False).rename(columns={"index": "__orig_index__"})

        # 3) 数值矩阵（一次性转换，后续全 numpy）
        self.X_all = (
            self.df_c[self.all_cols]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float32)
        )
        self.X_ref_all = (
            self.df_ref[self.all_cols]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float32)
        )

        self.n_rows = self.X_all.shape[0]
        self.all_pos = np.arange(len(self.all_cols), dtype=np.int32)
        col_to_pos = {c: i for i, c in enumerate(self.all_cols)}
        self.target_pos = np.array([col_to_pos[c] for c in self.target_cols], dtype=np.int32)
        self.fc_pos_all = np.array([col_to_pos[c] for c in self.fc_cols], dtype=np.int32)

        # 4) Judge 编码
        code_map = {"BEST": 0, "PASS": 1, "MEDIUM": 2}
        self.judge_code = self.df_c["Judge"].map(code_map).to_numpy(dtype=np.int8)
        self.pool = {j: np.where(self.judge_code == j)[0] for j in (0, 1, 2)}

        # 5) 目标统计量（优化列）
        X_ref_target = self.X_ref_all[:, self.target_pos]
        self.t_q1, self.t_med, self.t_q3, self.t_std = self._compute_stats(X_ref_target)
        self.t_shape = np.nanpercentile(X_ref_target, self.shape_q, axis=0)
        t_iqr = self.t_q3 - self.t_q1
        self.t_scale = np.where(t_iqr > 1e-6, t_iqr, np.where(self.t_std > 1e-6, self.t_std, 1.0))

        self.t_bimodal = np.array(
            [self._is_bimodal(X_ref_target[:, i]) for i in range(X_ref_target.shape[1])],
            dtype=bool,
        )

        # 6) 全 93 列参考统计（最终报告用）
        self.ref_q1_all, self.ref_med_all, self.ref_q3_all, self.ref_std_all = self._compute_stats(self.X_ref_all)
        self.ref_shape_all = np.nanpercentile(self.X_ref_all, self.shape_q, axis=0)
        iqr_all = self.ref_q3_all - self.ref_q1_all
        self.scale_all = np.where(
            iqr_all > 1e-6, iqr_all, np.where(self.ref_std_all > 1e-6, self.ref_std_all, 1.0)
        )
        self.ref_bimodal_all = np.array(
            [self._is_bimodal(self.X_ref_all[:, j]) for j in self.all_pos],
            dtype=bool,
        )

        # 7) 固定一个可行样本量 + 三类固定计数（便于同类交换加速）
        self.fixed_n, self.fixed_counts = self._find_feasible_size_and_counts()

    def _find_feasible_size_and_counts(self) -> Tuple[int, Dict[int, int]]:
        avail_best = len(self.pool[0])
        avail_pass = len(self.pool[1])
        avail_medium = len(self.pool[2])

        size_candidates = list(range(self.size_min, self.size_max + 1))
        size_candidates.sort(key=lambda n: (abs(n - self.target_size), n))

        best_solution = None  # (size_distance, ratio_deviation, n, counts_dict)

        for n in size_candidates:
            b_lo, b_hi = math.ceil(0.45 * n), math.floor(0.50 * n)
            p_lo, p_hi = math.ceil(0.42 * n), math.floor(0.47 * n)
            m_lo = math.ceil((0.08 - self.judge_tolerance) * n)
            m_hi = math.floor((0.08 + self.judge_tolerance) * n)
            m_lo = max(0, m_lo)

            if b_lo > b_hi or p_lo > p_hi or m_lo > m_hi:
                continue

            local_best = None
            for nb in range(b_lo, b_hi + 1):
                if nb > avail_best:
                    continue
                for npass in range(p_lo, p_hi + 1):
                    if npass > avail_pass:
                        continue
                    nmed = n - nb - npass
                    if nmed < m_lo or nmed > m_hi:
                        continue
                    if nmed > avail_medium:
                        continue

                    dev = abs(nb / n - 0.475) + abs(npass / n - 0.445) + abs(nmed / n - 0.08)
                    cand = (dev, {0: nb, 1: npass, 2: nmed})
                    if (local_best is None) or (cand[0] < local_best[0]):
                        local_best = cand

            if local_best is not None:
                size_distance = abs(n - self.target_size)
                solution = (size_distance, local_best[0], n, local_best[1])
                if (best_solution is None) or (solution < best_solution):
                    best_solution = solution

                # size_candidates 已按离 target_size 的距离排序，找到可行值可提前结束
                break

        if best_solution is None:
            raise ValueError(
                "在 320~340 范围内找不到满足 Judge 比例约束的可行解。请检查 df_C 的 Judge 数量分布。"
            )

        return best_solution[2], best_solution[3]

    def _sample_initial_indices(self) -> np.ndarray:
        nb = self.fixed_counts[0]
        npa = self.fixed_counts[1]
        nmed = self.fixed_counts[2]

        idx_b = self.rng.choice(self.pool[0], size=nb, replace=False)
        idx_p = self.rng.choice(self.pool[1], size=npa, replace=False)
        idx_m = self.rng.choice(self.pool[2], size=nmed, replace=False)

        idx = np.concatenate([idx_b, idx_p, idx_m])
        self.rng.shuffle(idx)
        return idx

    def _judge_ok(self, cnt: np.ndarray, n: int) -> Tuple[bool, Tuple[float, float, float], float]:
        bp = cnt[0] / n
        pp = cnt[1] / n
        mp = cnt[2] / n

        ok = (
            (0.45 <= bp <= 0.50)
            and (0.42 <= pp <= 0.47)
            and (abs(mp - 0.08) <= self.judge_tolerance)
        )

        dev = 0.0
        dev += max(0.0, 0.45 - bp, bp - 0.50)
        dev += max(0.0, 0.42 - pp, pp - 0.47)
        dev += max(0.0, abs(mp - 0.08) - self.judge_tolerance)

        return ok, (bp, pp, mp), dev

    def _evaluate(self, idx: np.ndarray, detail: bool = False) -> Dict:
        n = idx.size

        # --- Target columns score ---
        X_t = self.X_all[np.ix_(idx, self.target_pos)]  # shape [n, n_target]

        q = np.nanpercentile(X_t, [25, 50, 75], axis=0)
        q1, med, q3 = q[0], q[1], q[2]
        std = np.nanstd(X_t, axis=0, ddof=1)

        d_med = np.abs(med - self.t_med)
        d_q1 = np.abs(q1 - self.t_q1)
        d_q3 = np.abs(q3 - self.t_q3)
        d_std = np.abs(std - self.t_std)

        d_med = np.nan_to_num(d_med, nan=1e3, posinf=1e3, neginf=1e3)
        d_q1 = np.nan_to_num(d_q1, nan=1e3, posinf=1e3, neginf=1e3)
        d_q3 = np.nan_to_num(d_q3, nan=1e3, posinf=1e3, neginf=1e3)
        d_std = np.nan_to_num(d_std, nan=1e3, posinf=1e3, neginf=1e3)

        d_stack = np.vstack([d_med, d_q1, d_q3, d_std])
        d_max = np.max(d_stack, axis=0)
        stat_violation = d_max > self.stat_threshold
        stat_penalty = float(np.mean(np.mean(d_stack, axis=0)))
        stat_violation_rate = float(np.mean(stat_violation))

        # 分布形态相似（目标双峰列不参与形态约束）
        sub_shape = np.nanpercentile(X_t, self.shape_q, axis=0)
        shape_each = np.mean(np.abs(sub_shape - self.t_shape) / (self.t_scale + 1e-6), axis=0)
        shape_each = np.nan_to_num(shape_each, nan=1e3, posinf=1e3, neginf=1e3)
        shape_each[self.t_bimodal] = 0.0
        shape_penalty = float(np.mean(shape_each))

        # FC 列正态性软约束（总是按全 93 中 FC 列算）
        normality_penalty = 0.0
        if self.fc_pos_all.size > 0:
            X_fc = self.X_all[np.ix_(idx, self.fc_pos_all)]
            sk = skew(X_fc, axis=0, bias=False, nan_policy="omit")
            ku = kurtosis(X_fc, axis=0, fisher=False, bias=False, nan_policy="omit")
            sk = np.nan_to_num(sk, nan=10.0, posinf=10.0, neginf=10.0)
            ku = np.nan_to_num(ku, nan=10.0, posinf=10.0, neginf=10.0)

            normality_penalty = float(
                np.mean(
                    np.maximum(0.0, np.abs(sk) - self.normal_skew_thr)
                    + np.maximum(0.0, np.abs(ku - 3.0) - self.normal_kurt_thr)
                )
            )

        # --- Hard constraints ---
        size_ok = self.size_min <= n <= self.size_max
        size_dev = 0.0 if size_ok else abs(n - self.target_size) / max(1.0, self.target_size)

        cnt = np.bincount(self.judge_code[idx], minlength=3)
        judge_ok, judge_props, judge_dev = self._judge_ok(cnt, n)

        # 所有 93 列禁止双峰（硬约束）
        X_sub_all = self.X_all[idx, :]
        bimodal_flags = np.zeros(len(self.all_cols), dtype=bool)
        for j in self.all_pos:
            bimodal_flags[j] = self._is_bimodal(X_sub_all[:, j])
        bimodal_count = int(bimodal_flags.sum())

        hard_penalty = 0.0
        hard_count = 0
        if not size_ok:
            hard_penalty += 2e5 * (1.0 + size_dev)
            hard_count += 1
        if not judge_ok:
            hard_penalty += 4e5 * (1.0 + judge_dev)
            hard_count += 1
        if bimodal_count > 0:
            hard_penalty += 3e5 * bimodal_count
            hard_count += 1

        score = (
            hard_penalty
            + self.w_shape * shape_penalty
            + self.w_stat * stat_penalty
            + self.w_norm * normality_penalty
            + self.w_stat_violation * stat_violation_rate
        )

        out = {
            "score": float(score),
            "hard_count": int(hard_count),
            "size_ok": bool(size_ok),
            "judge_ok": bool(judge_ok),
            "judge_props": judge_props,
            "bimodal_count": bimodal_count,
            "bimodal_flags": bimodal_flags,
            "stat_violation_count": int(np.sum(stat_violation)),
            "stat_violation_rate": stat_violation_rate,
            "avg_stat_diff": stat_penalty,
            "shape_penalty": shape_penalty,
            "normality_penalty": normality_penalty,
        }

        if detail:
            out["d_med"] = d_med
            out["d_q1"] = d_q1
            out["d_q3"] = d_q3
            out["d_std"] = d_std
            out["shape_each"] = shape_each
            out["stat_violation_mask"] = stat_violation

        return out

    def _propose_swap(self, selected: np.ndarray):
        # 同 Judge 类别内交换，保持 judge 比例严格不变
        j = int(self.rng.integers(0, 3))
        pool_j = self.pool[j]

        selected_j = pool_j[selected[pool_j]]
        unselected_j = pool_j[~selected[pool_j]]

        if selected_j.size == 0 or unselected_j.size == 0:
            return None

        out_i = int(selected_j[self.rng.integers(0, selected_j.size)])
        in_i = int(unselected_j[self.rng.integers(0, unselected_j.size)])

        selected[out_i] = False
        selected[in_i] = True
        return out_i, in_i

    def _optimize_once(self, init_idx: np.ndarray):
        selected = np.zeros(self.n_rows, dtype=bool)
        selected[init_idx] = True

        current_idx = np.flatnonzero(selected)
        cur_eval = self._evaluate(current_idx)

        best_idx = current_idx.copy()
        best_eval = dict(cur_eval)

        no_improve = 0
        run_history = []

        for it in range(self.max_iter):
            # 1~2 个交换动作
            n_swaps = 1 if self.rng.random() < 0.85 else 2
            swaps = []
            for _ in range(n_swaps):
                mv = self._propose_swap(selected)
                if mv is not None:
                    swaps.append(mv)

            if not swaps:
                continue

            cand_idx = np.flatnonzero(selected)
            cand_eval = self._evaluate(cand_idx)

            temp = max(1e-9, self.init_temp * (self.cooling ** it))
            accept = False

            if cand_eval["score"] <= cur_eval["score"]:
                accept = True
            else:
                delta = cand_eval["score"] - cur_eval["score"]
                # 防止极大 delta 导致无意义 exp 计算
                if delta < 50.0:
                    accept = self.rng.random() < math.exp(-delta / temp)

            if accept:
                current_idx = cand_idx
                cur_eval = cand_eval

                if cand_eval["score"] + 1e-12 < best_eval["score"]:
                    best_idx = current_idx.copy()
                    best_eval = dict(cand_eval)
                    no_improve = 0
                else:
                    no_improve += 1
            else:
                # 回滚交换
                for out_i, in_i in reversed(swaps):
                    selected[out_i] = True
                    selected[in_i] = False
                no_improve += 1

            if (it + 1) % 25 == 0:
                run_history.append(
                    {
                        "iter": it + 1,
                        "score": cur_eval["score"],
                        "best_score": best_eval["score"],
                        "bimodal_count": cur_eval["bimodal_count"],
                        "stat_violation_count": cur_eval["stat_violation_count"],
                        "avg_stat_diff": cur_eval["avg_stat_diff"],
                        "shape_penalty": cur_eval["shape_penalty"],
                        "normality_penalty": cur_eval["normality_penalty"],
                    }
                )

            if no_improve >= self.early_stop:
                break

        return best_idx, best_eval, run_history

    def _build_column_report(self, best_idx: np.ndarray) -> pd.DataFrame:
        X_sub = self.X_all[best_idx, :]

        sub_q1, sub_med, sub_q3, sub_std = self._compute_stats(X_sub)
        sub_shape_all = np.nanpercentile(X_sub, self.shape_q, axis=0)
        shape_dist = np.mean(
            np.abs(sub_shape_all - self.ref_shape_all) / (self.scale_all + 1e-6), axis=0
        )
        shape_dist = np.nan_to_num(shape_dist, nan=1e3, posinf=1e3, neginf=1e3)

        sub_bimodal = np.array([self._is_bimodal(X_sub[:, j]) for j in self.all_pos], dtype=bool)

        med_diff = np.abs(sub_med - self.ref_med_all)
        q1_diff = np.abs(sub_q1 - self.ref_q1_all)
        q3_diff = np.abs(sub_q3 - self.ref_q3_all)
        std_diff = np.abs(sub_std - self.ref_std_all)

        med_diff = np.nan_to_num(med_diff, nan=1e3, posinf=1e3, neginf=1e3)
        q1_diff = np.nan_to_num(q1_diff, nan=1e3, posinf=1e3, neginf=1e3)
        q3_diff = np.nan_to_num(q3_diff, nan=1e3, posinf=1e3, neginf=1e3)
        std_diff = np.nan_to_num(std_diff, nan=1e3, posinf=1e3, neginf=1e3)

        is_fc = np.array([str(c).startswith("FC") for c in self.all_cols], dtype=bool)
        target_set = set(self.target_cols)
        is_target = np.array([c in target_set for c in self.all_cols], dtype=bool)

        # 参考双峰标记（仅 target 列有该特殊规则，非 target 列置 False）
        target_bi_map = {c: bool(b) for c, b in zip(self.target_cols, self.t_bimodal)}
        special_rule = np.array([target_bi_map.get(c, False) for c in self.all_cols], dtype=bool)

        # FC 正态指标
        fc_skew = np.full(len(self.all_cols), np.nan, dtype=float)
        fc_kurt = np.full(len(self.all_cols), np.nan, dtype=float)
        for p in self.fc_pos_all:
            col = X_sub[:, p]
            y = col[np.isfinite(col)]
            if y.size >= 8:
                fc_skew[p] = float(skew(y, bias=False))
                fc_kurt[p] = float(kurtosis(y, fisher=False, bias=False))

        fc_norm_pen = np.full(len(self.all_cols), np.nan, dtype=float)
        for p in self.fc_pos_all:
            skv = fc_skew[p]
            kuv = fc_kurt[p]
            if np.isfinite(skv) and np.isfinite(kuv):
                fc_norm_pen[p] = max(0.0, abs(skv) - self.normal_skew_thr) + max(
                    0.0, abs(kuv - 3.0) - self.normal_kurt_thr
                )

        max_diff = np.maximum.reduce([med_diff, q1_diff, q3_diff, std_diff])
        above_thr = max_diff > self.stat_threshold

        report = pd.DataFrame(
            {
                "column": self.all_cols,
                "is_target": is_target,
                "is_fc": is_fc,
                "ref_bimodal": self.ref_bimodal_all,
                "target_special_rule_ref_bimodal": special_rule,
                "subset_bimodal": sub_bimodal,
                "shape_distance": shape_dist,
                "median_diff": med_diff,
                "q1_diff": q1_diff,
                "q3_diff": q3_diff,
                "std_diff": std_diff,
                "max_diff": max_diff,
                "above_stat_threshold": above_thr,
                "fc_skew": fc_skew,
                "fc_kurtosis": fc_kurt,
                "fc_normality_penalty": fc_norm_pen,
            }
        )

        return report

    def fit(self, df_C: pd.DataFrame, df_ref: pd.DataFrame):
        self._prepare(df_C=df_C, df_ref=df_ref)

        best_global_idx = None
        best_global_eval = None
        history_rows = []

        for run in range(1, self.n_starts + 1):
            init_idx = self._sample_initial_indices()
            best_idx, best_eval, run_hist = self._optimize_once(init_idx)

            for row in run_hist:
                row["run"] = run
                history_rows.append(row)

            if (best_global_eval is None) or (best_eval["score"] < best_global_eval["score"]):
                best_global_idx = best_idx
                best_global_eval = best_eval

        final_eval = self._evaluate(best_global_idx, detail=True)

        subset_df = self.df_c.iloc[best_global_idx].copy()
        subset_df = subset_df.sort_values("__orig_index__")
        subset_df.index = subset_df["__orig_index__"].to_numpy()
        subset_df = subset_df.drop(columns=["__orig_index__"])

        col_report = self._build_column_report(best_global_idx)
        history_df = pd.DataFrame(history_rows)

        bp, pp, mp = final_eval["judge_props"]
        summary = {
            "target_mode": self.target_mode,
            "n_target_columns": len(self.target_cols),
            "n_fc_columns": len(self.fc_cols),
            "subset_size": int(len(best_global_idx)),
            "size_ok": bool(final_eval["size_ok"]),
            "judge_ok": bool(final_eval["judge_ok"]),
            "judge_best": float(bp),
            "judge_pass": float(pp),
            "judge_medium": float(mp),
            "all_non_bimodal": bool(final_eval["bimodal_count"] == 0),
            "bimodal_columns_count": int(final_eval["bimodal_count"]),
            "stat_violation_count": int(final_eval["stat_violation_count"]),
            "avg_stat_diff": float(final_eval["avg_stat_diff"]),
            "shape_penalty": float(final_eval["shape_penalty"]),
            "fc_normality_penalty": float(final_eval["normality_penalty"]),
            "final_score": float(final_eval["score"]),
            "fixed_sample_size": int(self.fixed_n),
            "fixed_counts": {
                "BEST": int(self.fixed_counts[0]),
                "PASS": int(self.fixed_counts[1]),
                "MEDIUM": int(self.fixed_counts[2]),
            },
        }

        return subset_df, summary, col_report, history_df


# ---------------------------
# Example usage
# ---------------------------
if __name__ == "__main__":
    df_C = load_table(r"C:\Users\HP\Desktop\code\C.csv")

    # 规范化并计数
    hash_norm = df_C['#'].fillna('<NA>').astype(str).str.strip()
    counts = hash_norm.value_counts()
    counts_df = counts.rename_axis('#').reset_index(name='count').sort_values('count', ascending=False)

    # 保留 count > 320 的类别并取第一个
    classes_over_320 = counts[counts > 320].index.tolist()
    if classes_over_320:
        first_val = classes_over_320[0]
        C_data = df_C[hash_norm == first_val].copy()
        print(f"Selected first class: {first_val}, n={len(C_data)}")
    else:
        C_data = df_C.iloc[0:0].copy()
        print("No class with count > 320 found; C_data is empty.")

    df_C30 = load_table(r"C:\Users\HP\Desktop\code\C30.csv")
    data_ref = df_C30[df_C30["#"] == "0.0XM C3.0_CGS"].copy()
    if data_ref.empty:
        raise ValueError("df_C30 参考组为空，请检查过滤条件。")

    optimizer = FastSubsetOptimizer(
        target_mode="all_93",
        stat_threshold=3.0,
        size_min=320,
        size_max=340,
        target_size=330,
        judge_tolerance=0.015,
        n_starts=5,
        max_iter=800,
        early_stop=180,
        random_seed=42,
    )

    subset_df, summary, col_report, history_df = optimizer.fit(df_C=C_data, df_ref=data_ref)

    print("Summary:")
    for k, v in summary.items():
        print(f"  {k}: {v}")

    subset_df.to_excel(r"C:\Users\HP\Desktop\code\optimal_subset.xlsx", index=False)
    col_report.to_excel(r"C:\Users\HP\Desktop\code\col_report.xlsx", index=False)
    history_df.to_excel(r"C:\Users\HP\Desktop\code\search_history.xlsx", index=False)

Selected first class: 5C-C5, n=1727
Summary:
  target_mode: all_93
  n_target_columns: 93
  n_fc_columns: 22
  subset_size: 330
  size_ok: True
  judge_ok: True
  judge_best: 0.47575757575757577
  judge_pass: 0.44545454545454544
  judge_medium: 0.07878787878787878
  all_non_bimodal: True
  bimodal_columns_count: 0
  stat_violation_count: 9
  avg_stat_diff: 0.9825717245237351
  shape_penalty: 0.5492656677202398
  fc_normality_penalty: 1.0873478651046753
  final_score: 2.3134540835605053
  fixed_sample_size: 330
  fixed_counts: {'BEST': 157, 'PASS': 147, 'MEDIUM': 26}


In [22]:
import math
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew

warnings.filterwarnings("ignore", category=RuntimeWarning)


def load_table(path: str) -> pd.DataFrame:
    p = str(path).lower()
    if p.endswith(".csv"):
        return pd.read_csv(path, low_memory=False)
    return pd.read_excel(path, engine="openpyxl")


class FastSubsetOptimizer:
    """
    从 df_C 中抽取 320~340（默认约330）样本，使其分布和统计量尽量接近 df_ref。
    约束：
    - Judge 比例强制满足
    - 所有 93 列不能双峰（增强版：峰间谷深 + 两侧质量）
    - FC 列尽量接近正态
    - 尾部宽度、IQR外比例、极端点比例要接近参考
    """

    def __init__(
        self,
        target_mode: str = "all_93",  # "all_93" or "fc_only"
        size_min: int = 320,
        size_max: int = 340,
        target_size: int = 330,
        judge_tolerance: float = 0.015,
        stat_threshold: float = 3.0,
        n_starts: int = 5,
        max_iter: int = 800,
        early_stop: int = 180,
        init_temp: float = 0.08,
        cooling: float = 0.997,
        normal_skew_thr: float = 0.5,
        normal_kurt_thr: float = 1.5,
        random_seed: int = 42,
        w_shape: float = 1.2,
        w_stat: float = 1.0,
        w_norm: float = 0.6,
        w_stat_violation: float = 0.2,
        # 新增：尾部与异常值权重
        w_tail: float = 1.2,
        w_outlier: float = 2.5,
        w_extreme: float = 8.0,
        # 新增：极端值硬约束余量
        hard_extreme_margin: float = 0.02,
        # 新增：双峰判定参数（更敏感于中间凹陷）
        bimodal_min_n: int = 30,
        bimodal_peak_height_ratio: float = 0.10,
        bimodal_sep_ratio: float = 0.08,
        bimodal_valley_ratio: float = 0.78,
        bimodal_min_side_mass: float = 0.12,
    ):
        self.target_mode = target_mode
        self.size_min = size_min
        self.size_max = size_max
        self.target_size = target_size
        self.judge_tolerance = judge_tolerance
        self.stat_threshold = stat_threshold

        self.n_starts = n_starts
        self.max_iter = max_iter
        self.early_stop = early_stop
        self.init_temp = init_temp
        self.cooling = cooling

        self.normal_skew_thr = normal_skew_thr
        self.normal_kurt_thr = normal_kurt_thr

        self.w_shape = w_shape
        self.w_stat = w_stat
        self.w_norm = w_norm
        self.w_stat_violation = w_stat_violation
        self.w_tail = w_tail
        self.w_outlier = w_outlier
        self.w_extreme = w_extreme

        self.hard_extreme_margin = hard_extreme_margin

        self.bimodal_min_n = bimodal_min_n
        self.bimodal_peak_height_ratio = bimodal_peak_height_ratio
        self.bimodal_sep_ratio = bimodal_sep_ratio
        self.bimodal_valley_ratio = bimodal_valley_ratio
        self.bimodal_min_side_mass = bimodal_min_side_mass

        self.rng = np.random.default_rng(random_seed)

        # Populated in _prepare
        self.df_c = None
        self.df_ref = None
        self.all_cols = None
        self.fc_cols = None
        self.target_cols = None

        self.X_all = None
        self.X_ref_all = None
        self.judge_code = None
        self.pool = None
        self.n_rows = 0

        self.all_pos = None
        self.target_pos = None
        self.fc_pos_all = None

        # Target stats (for optimization columns)
        self.t_q1 = None
        self.t_med = None
        self.t_q3 = None
        self.t_std = None
        self.t_shape = None
        self.t_scale = None
        self.t_bimodal = None

        # 新增：target 列尾部与异常值参考
        self.ref_p01 = None
        self.ref_p99 = None
        self.ref_iqr = None
        self.tail_scale = None
        self.ref_low_15 = None
        self.ref_up_15 = None
        self.ref_low_30 = None
        self.ref_up_30 = None
        self.ref_out_rate_15 = None
        self.ref_out_rate_30 = None

        # All-93 reference stats (for final report)
        self.ref_q1_all = None
        self.ref_med_all = None
        self.ref_q3_all = None
        self.ref_std_all = None
        self.ref_shape_all = None
        self.scale_all = None
        self.ref_bimodal_all = None

        # Fixed composition for fast swap search
        self.fixed_n = None
        self.fixed_counts = None  # dict {0: best_n, 1: pass_n, 2: medium_n}

        self.shape_q = np.array([5, 15, 25, 35, 50, 65, 75, 85, 95], dtype=np.float64)

    @staticmethod
    def _extract_93_columns(df: pd.DataFrame) -> List[str]:
        if "EFL" not in df.columns:
            raise ValueError("未找到列 EFL，无法定位目标 93 列。")
        start = df.columns.get_loc("EFL")
        cols = list(df.columns[start:start + 93])
        if len(cols) != 93:
            raise ValueError(f"从 EFL 开始不足 93 列，实际只有 {len(cols)} 列。")
        return cols

    @staticmethod
    def _normalize_judge(s: pd.Series) -> pd.Series:
        # 兼容 BSET -> BEST
        x = s.astype(str).str.strip().str.upper()
        x = x.replace({"BSET": "BEST"})
        return x

    @staticmethod
    def _compute_stats(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        # X shape: [n_samples, n_cols]
        q = np.nanpercentile(X, [25, 50, 75], axis=0)
        q1, med, q3 = q[0], q[1], q[2]
        std = np.nanstd(X, axis=0, ddof=1)
        std = np.nan_to_num(std, nan=0.0, posinf=0.0, neginf=0.0)
        return q1, med, q3, std

    @staticmethod
    def _is_bimodal(
        x: np.ndarray,
        min_n: int = 30,
        peak_height_ratio: float = 0.10,
        sep_ratio: float = 0.08,
        valley_ratio: float = 0.78,
        min_side_mass: float = 0.12,
    ) -> bool:
        """
        增强版双峰检测（速度优先）：
        - 直方图 + 平滑 + 双峰峰值
        - 峰间谷深（valley / min(peak1,peak2)）
        - 谷点两侧质量不能太小
        """
        y = np.asarray(x, dtype=np.float64)
        y = y[np.isfinite(y)]
        n = y.size
        if n < min_n:
            return False

        y_min = float(np.min(y))
        y_max = float(np.max(y))
        if (not np.isfinite(y_min)) or (not np.isfinite(y_max)) or (y_max <= y_min):
            return False

        bins = int(np.clip(np.sqrt(n), 12, 48))
        hist, edges = np.histogram(y, bins=bins)
        if hist.max() <= 0:
            return False

        smooth = np.convolve(hist.astype(np.float64), np.array([1.0, 2.0, 1.0]), mode="same")
        thr = peak_height_ratio * smooth.max()

        left = smooth[:-2]
        mid = smooth[1:-1]
        right = smooth[2:]
        peak_idx = np.where((mid > left) & (mid >= right) & (mid >= thr))[0] + 1
        if peak_idx.size < 2:
            return False

        # 取最高两峰
        top2 = peak_idx[np.argsort(smooth[peak_idx])[-2:]]
        p1, p2 = int(min(top2)), int(max(top2))

        centers = 0.5 * (edges[:-1] + edges[1:])
        sep = abs(centers[p1] - centers[p2]) / (y_max - y_min + 1e-12)
        if sep < sep_ratio:
            return False

        # 峰间谷深
        valley_pos = p1 + int(np.argmin(smooth[p1:p2 + 1]))
        valley = float(smooth[valley_pos])
        min_peak = float(min(smooth[p1], smooth[p2])) + 1e-12
        if valley > valley_ratio * min_peak:
            return False

        # 谷点左右质量，防止“一个主峰 + 一点噪声”误判双峰
        left_mass = float(hist[:valley_pos + 1].sum()) / max(1, n)
        right_mass = float(hist[valley_pos + 1:].sum()) / max(1, n)
        if min(left_mass, right_mass) < min_side_mass:
            return False

        return True

    def _is_bimodal_cfg(self, x: np.ndarray) -> bool:
        return self._is_bimodal(
            x=x,
            min_n=self.bimodal_min_n,
            peak_height_ratio=self.bimodal_peak_height_ratio,
            sep_ratio=self.bimodal_sep_ratio,
            valley_ratio=self.bimodal_valley_ratio,
            min_side_mass=self.bimodal_min_side_mass,
        )

    def _prepare(self, df_C: pd.DataFrame, df_ref: pd.DataFrame) -> None:
        self.df_c = df_C.copy()
        self.df_ref = df_ref.copy()

        # 1) 93 列与 FC 列
        self.all_cols = self._extract_93_columns(self.df_ref)
        missing = [c for c in self.all_cols if c not in self.df_c.columns]
        if missing:
            raise ValueError(f"df_C 缺少目标列: {missing[:5]} ...")

        self.fc_cols = [c for c in self.all_cols if str(c).startswith("FC")]
        if self.target_mode == "all_93":
            self.target_cols = list(self.all_cols)
        elif self.target_mode == "fc_only":
            if not self.fc_cols:
                raise ValueError("target_mode='fc_only' 但未找到 FC 列。")
            self.target_cols = list(self.fc_cols)
        else:
            raise ValueError("target_mode 只能是 'all_93' 或 'fc_only'。")

        # 2) Judge 标准化并过滤
        if "Judge" not in self.df_c.columns:
            raise ValueError("df_C 缺少 Judge 列。")
        self.df_c["Judge"] = self._normalize_judge(self.df_c["Judge"])
        valid_mask = self.df_c["Judge"].isin(["BEST", "PASS", "MEDIUM"])
        self.df_c = self.df_c.loc[valid_mask].copy()
        if self.df_c.empty:
            raise ValueError("df_C 在过滤 Judge 后为空。")

        # 保存原始索引
        self.df_c = self.df_c.reset_index(drop=False).rename(columns={"index": "__orig_index__"})

        # 3) 数值矩阵（一次性转换，后续全 numpy）
        self.X_all = (
            self.df_c[self.all_cols]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float32)
        )
        self.X_ref_all = (
            self.df_ref[self.all_cols]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float32)
        )

        self.n_rows = self.X_all.shape[0]
        self.all_pos = np.arange(len(self.all_cols), dtype=np.int32)
        col_to_pos = {c: i for i, c in enumerate(self.all_cols)}
        self.target_pos = np.array([col_to_pos[c] for c in self.target_cols], dtype=np.int32)
        self.fc_pos_all = np.array([col_to_pos[c] for c in self.fc_cols], dtype=np.int32)

        # 4) Judge 编码
        code_map = {"BEST": 0, "PASS": 1, "MEDIUM": 2}
        self.judge_code = self.df_c["Judge"].map(code_map).to_numpy(dtype=np.int8)
        self.pool = {j: np.where(self.judge_code == j)[0] for j in (0, 1, 2)}

        # 5) 目标统计量（优化列）
        X_ref_target = self.X_ref_all[:, self.target_pos]
        self.t_q1, self.t_med, self.t_q3, self.t_std = self._compute_stats(X_ref_target)
        self.t_shape = np.nanpercentile(X_ref_target, self.shape_q, axis=0)
        t_iqr = self.t_q3 - self.t_q1
        self.t_scale = np.where(t_iqr > 1e-6, t_iqr, np.where(self.t_std > 1e-6, self.t_std, 1.0))

        # 增强双峰标记（目标列）
        self.t_bimodal = np.array(
            [self._is_bimodal_cfg(X_ref_target[:, i]) for i in range(X_ref_target.shape[1])],
            dtype=bool,
        )

        # 新增：target 列尾部与异常率参考
        self.ref_p01 = np.nanpercentile(X_ref_target, 1, axis=0)
        self.ref_p99 = np.nanpercentile(X_ref_target, 99, axis=0)
        self.ref_p01 = np.nan_to_num(self.ref_p01, nan=0.0, posinf=0.0, neginf=0.0)
        self.ref_p99 = np.nan_to_num(self.ref_p99, nan=0.0, posinf=0.0, neginf=0.0)

        self.ref_iqr = self.t_q3 - self.t_q1
        self.ref_iqr = np.nan_to_num(self.ref_iqr, nan=0.0, posinf=0.0, neginf=0.0)

        # 尾部宽度归一化尺度
        tail_ref = self.ref_p99 - self.ref_p01
        self.tail_scale = np.where(
            tail_ref > 1e-6,
            tail_ref,
            np.where(self.ref_iqr > 1e-6, self.ref_iqr, np.where(self.t_std > 1e-6, self.t_std, 1.0)),
        )

        iqr_safe = np.where(
            self.ref_iqr > 1e-6,
            self.ref_iqr,
            np.where(self.t_std > 1e-6, self.t_std, 1.0),
        )
        self.ref_low_15 = self.t_q1 - 1.5 * iqr_safe
        self.ref_up_15 = self.t_q3 + 1.5 * iqr_safe
        self.ref_low_30 = self.t_q1 - 3.0 * iqr_safe
        self.ref_up_30 = self.t_q3 + 3.0 * iqr_safe

        ref_out_15 = (X_ref_target < self.ref_low_15) | (X_ref_target > self.ref_up_15)
        ref_out_30 = (X_ref_target < self.ref_low_30) | (X_ref_target > self.ref_up_30)
        self.ref_out_rate_15 = np.nanmean(ref_out_15, axis=0)
        self.ref_out_rate_30 = np.nanmean(ref_out_30, axis=0)
        self.ref_out_rate_15 = np.nan_to_num(self.ref_out_rate_15, nan=0.0, posinf=0.0, neginf=0.0)
        self.ref_out_rate_30 = np.nan_to_num(self.ref_out_rate_30, nan=0.0, posinf=0.0, neginf=0.0)

        # 6) 全 93 列参考统计（最终报告用）
        self.ref_q1_all, self.ref_med_all, self.ref_q3_all, self.ref_std_all = self._compute_stats(self.X_ref_all)
        self.ref_shape_all = np.nanpercentile(self.X_ref_all, self.shape_q, axis=0)
        iqr_all = self.ref_q3_all - self.ref_q1_all
        self.scale_all = np.where(
            iqr_all > 1e-6, iqr_all, np.where(self.ref_std_all > 1e-6, self.ref_std_all, 1.0)
        )
        self.ref_bimodal_all = np.array(
            [self._is_bimodal_cfg(self.X_ref_all[:, j]) for j in self.all_pos],
            dtype=bool,
        )

        # 7) 固定一个可行样本量 + 三类固定计数（便于同类交换加速）
        self.fixed_n, self.fixed_counts = self._find_feasible_size_and_counts()

    def _find_feasible_size_and_counts(self) -> Tuple[int, Dict[int, int]]:
        avail_best = len(self.pool[0])
        avail_pass = len(self.pool[1])
        avail_medium = len(self.pool[2])

        size_candidates = list(range(self.size_min, self.size_max + 1))
        size_candidates.sort(key=lambda n: (abs(n - self.target_size), n))

        best_solution = None  # (size_distance, ratio_deviation, n, counts_dict)

        for n in size_candidates:
            b_lo, b_hi = math.ceil(0.45 * n), math.floor(0.50 * n)
            p_lo, p_hi = math.ceil(0.42 * n), math.floor(0.47 * n)
            m_lo = math.ceil((0.08 - self.judge_tolerance) * n)
            m_hi = math.floor((0.08 + self.judge_tolerance) * n)
            m_lo = max(0, m_lo)

            if b_lo > b_hi or p_lo > p_hi or m_lo > m_hi:
                continue

            local_best = None
            for nb in range(b_lo, b_hi + 1):
                if nb > avail_best:
                    continue
                for npass in range(p_lo, p_hi + 1):
                    if npass > avail_pass:
                        continue
                    nmed = n - nb - npass
                    if nmed < m_lo or nmed > m_hi:
                        continue
                    if nmed > avail_medium:
                        continue

                    dev = abs(nb / n - 0.475) + abs(npass / n - 0.445) + abs(nmed / n - 0.08)
                    cand = (dev, {0: nb, 1: npass, 2: nmed})
                    if (local_best is None) or (cand[0] < local_best[0]):
                        local_best = cand

            if local_best is not None:
                size_distance = abs(n - self.target_size)
                solution = (size_distance, local_best[0], n, local_best[1])
                if (best_solution is None) or (solution < best_solution):
                    best_solution = solution

                # size_candidates 已按离 target_size 的距离排序，找到可行值可提前结束
                break

        if best_solution is None:
            raise ValueError(
                "在 320~340 范围内找不到满足 Judge 比例约束的可行解。请检查 df_C 的 Judge 数量分布。"
            )

        return best_solution[2], best_solution[3]

    def _sample_initial_indices(self) -> np.ndarray:
        nb = self.fixed_counts[0]
        npa = self.fixed_counts[1]
        nmed = self.fixed_counts[2]

        idx_b = self.rng.choice(self.pool[0], size=nb, replace=False)
        idx_p = self.rng.choice(self.pool[1], size=npa, replace=False)
        idx_m = self.rng.choice(self.pool[2], size=nmed, replace=False)

        idx = np.concatenate([idx_b, idx_p, idx_m])
        self.rng.shuffle(idx)
        return idx

    def _judge_ok(self, cnt: np.ndarray, n: int) -> Tuple[bool, Tuple[float, float, float], float]:
        bp = cnt[0] / n
        pp = cnt[1] / n
        mp = cnt[2] / n

        ok = (
            (0.45 <= bp <= 0.50)
            and (0.42 <= pp <= 0.47)
            and (abs(mp - 0.08) <= self.judge_tolerance)
        )

        dev = 0.0
        dev += max(0.0, 0.45 - bp, bp - 0.50)
        dev += max(0.0, 0.42 - pp, pp - 0.47)
        dev += max(0.0, abs(mp - 0.08) - self.judge_tolerance)

        return ok, (bp, pp, mp), dev

    def _evaluate(self, idx: np.ndarray, detail: bool = False) -> Dict:
        n = idx.size

        # --- Target columns score ---
        X_t = self.X_all[np.ix_(idx, self.target_pos)]  # [n, n_target]

        q = np.nanpercentile(X_t, [25, 50, 75], axis=0)
        q1, med, q3 = q[0], q[1], q[2]
        std = np.nanstd(X_t, axis=0, ddof=1)

        d_med = np.abs(med - self.t_med)
        d_q1 = np.abs(q1 - self.t_q1)
        d_q3 = np.abs(q3 - self.t_q3)
        d_std = np.abs(std - self.t_std)

        d_med = np.nan_to_num(d_med, nan=1e3, posinf=1e3, neginf=1e3)
        d_q1 = np.nan_to_num(d_q1, nan=1e3, posinf=1e3, neginf=1e3)
        d_q3 = np.nan_to_num(d_q3, nan=1e3, posinf=1e3, neginf=1e3)
        d_std = np.nan_to_num(d_std, nan=1e3, posinf=1e3, neginf=1e3)

        d_stack = np.vstack([d_med, d_q1, d_q3, d_std])
        d_max = np.max(d_stack, axis=0)
        stat_violation = d_max > self.stat_threshold
        stat_penalty = float(np.mean(np.mean(d_stack, axis=0)))
        stat_violation_rate = float(np.mean(stat_violation))

        # 形态相似（目标双峰列不参与形态约束）
        sub_shape = np.nanpercentile(X_t, self.shape_q, axis=0)
        shape_each = np.mean(np.abs(sub_shape - self.t_shape) / (self.t_scale + 1e-6), axis=0)
        shape_each = np.nan_to_num(shape_each, nan=1e3, posinf=1e3, neginf=1e3)
        shape_each[self.t_bimodal] = 0.0
        shape_penalty = float(np.mean(shape_each))

        # 新增：尾部宽度惩罚（p99-p1）
        p01 = np.nanpercentile(X_t, 1, axis=0)
        p99 = np.nanpercentile(X_t, 99, axis=0)
        tail_sub = p99 - p01
        tail_ref = self.ref_p99 - self.ref_p01
        tail_penalty_each = np.abs(tail_sub - tail_ref) / (self.tail_scale + 1e-6)
        tail_penalty_each = np.nan_to_num(tail_penalty_each, nan=1e3, posinf=1e3, neginf=1e3)
        tail_penalty = float(np.mean(tail_penalty_each))

        # 新增：IQR外比例（1.5IQR）与极端比例（3IQR）
        out_15 = (X_t < self.ref_low_15) | (X_t > self.ref_up_15)
        out_30 = (X_t < self.ref_low_30) | (X_t > self.ref_up_30)

        out_rate_15 = np.nanmean(out_15, axis=0)
        out_rate_30 = np.nanmean(out_30, axis=0)
        out_rate_15 = np.nan_to_num(out_rate_15, nan=0.0, posinf=0.0, neginf=0.0)
        out_rate_30 = np.nan_to_num(out_rate_30, nan=0.0, posinf=0.0, neginf=0.0)

        outlier_penalty_each = np.abs(out_rate_15 - self.ref_out_rate_15)
        outlier_penalty_each = np.nan_to_num(outlier_penalty_each, nan=1e3, posinf=1e3, neginf=1e3)
        outlier_penalty = float(np.mean(outlier_penalty_each))

        extreme_excess_each = np.maximum(0.0, out_rate_30 - (self.ref_out_rate_30 + 0.01))
        extreme_penalty = float(np.mean(extreme_excess_each))

        # FC 正态性软约束
        normality_penalty = 0.0
        if self.fc_pos_all.size > 0:
            X_fc = self.X_all[np.ix_(idx, self.fc_pos_all)]
            sk = skew(X_fc, axis=0, bias=False, nan_policy="omit")
            ku = kurtosis(X_fc, axis=0, fisher=False, bias=False, nan_policy="omit")
            sk = np.nan_to_num(sk, nan=10.0, posinf=10.0, neginf=10.0)
            ku = np.nan_to_num(ku, nan=10.0, posinf=10.0, neginf=10.0)

            normality_penalty = float(
                np.mean(
                    np.maximum(0.0, np.abs(sk) - self.normal_skew_thr)
                    + np.maximum(0.0, np.abs(ku - 3.0) - self.normal_kurt_thr)
                )
            )

        # --- Hard constraints ---
        size_ok = self.size_min <= n <= self.size_max
        size_dev = 0.0 if size_ok else abs(n - self.target_size) / max(1.0, self.target_size)

        cnt = np.bincount(self.judge_code[idx], minlength=3)
        judge_ok, judge_props, judge_dev = self._judge_ok(cnt, n)

        # 所有 93 列禁止双峰（硬约束）
        X_sub_all = self.X_all[idx, :]
        bimodal_flags = np.zeros(len(self.all_cols), dtype=bool)
        for j in self.all_pos:
            bimodal_flags[j] = self._is_bimodal_cfg(X_sub_all[:, j])
        bimodal_count = int(bimodal_flags.sum())

        # 极端值硬约束：如果 target 列中某列 3IQR 外比例明显高于参考
        extreme_hard_mask = out_rate_30 > (self.ref_out_rate_30 + self.hard_extreme_margin)
        extreme_hard_count = int(np.sum(extreme_hard_mask))

        hard_penalty = 0.0
        hard_count = 0
        if not size_ok:
            hard_penalty += 2e5 * (1.0 + size_dev)
            hard_count += 1
        if not judge_ok:
            hard_penalty += 4e5 * (1.0 + judge_dev)
            hard_count += 1
        if bimodal_count > 0:
            hard_penalty += 3e5 * bimodal_count
            hard_count += 1
        if extreme_hard_count > 0:
            hard_penalty += 2e5 * extreme_hard_count
            hard_count += 1

        score = (
            hard_penalty
            + self.w_shape * shape_penalty
            + self.w_stat * stat_penalty
            + self.w_norm * normality_penalty
            + self.w_stat_violation * stat_violation_rate
            + self.w_tail * tail_penalty
            + self.w_outlier * outlier_penalty
            + self.w_extreme * extreme_penalty
        )

        out = {
            "score": float(score),
            "hard_count": int(hard_count),
            "size_ok": bool(size_ok),
            "judge_ok": bool(judge_ok),
            "judge_props": judge_props,
            "bimodal_count": bimodal_count,
            "bimodal_flags": bimodal_flags,
            "extreme_hard_count": extreme_hard_count,
            "stat_violation_count": int(np.sum(stat_violation)),
            "stat_violation_rate": stat_violation_rate,
            "avg_stat_diff": stat_penalty,
            "shape_penalty": shape_penalty,
            "tail_penalty": tail_penalty,
            "outlier_penalty": outlier_penalty,
            "extreme_penalty": extreme_penalty,
            "normality_penalty": normality_penalty,
        }

        if detail:
            out["d_med"] = d_med
            out["d_q1"] = d_q1
            out["d_q3"] = d_q3
            out["d_std"] = d_std
            out["shape_each"] = shape_each
            out["tail_penalty_each"] = tail_penalty_each
            out["out_rate_15"] = out_rate_15
            out["out_rate_30"] = out_rate_30
            out["extreme_excess_each"] = extreme_excess_each
            out["stat_violation_mask"] = stat_violation

        return out

    def _propose_swap(self, selected: np.ndarray):
        # 同 Judge 类别内交换，保持 judge 比例严格不变
        j = int(self.rng.integers(0, 3))
        pool_j = self.pool[j]

        selected_j = pool_j[selected[pool_j]]
        unselected_j = pool_j[~selected[pool_j]]

        if selected_j.size == 0 or unselected_j.size == 0:
            return None

        out_i = int(selected_j[self.rng.integers(0, selected_j.size)])
        in_i = int(unselected_j[self.rng.integers(0, unselected_j.size)])

        selected[out_i] = False
        selected[in_i] = True
        return out_i, in_i

    def _optimize_once(self, init_idx: np.ndarray):
        selected = np.zeros(self.n_rows, dtype=bool)
        selected[init_idx] = True

        current_idx = np.flatnonzero(selected)
        cur_eval = self._evaluate(current_idx)

        best_idx = current_idx.copy()
        best_eval = dict(cur_eval)

        no_improve = 0
        run_history = []

        for it in range(self.max_iter):
            # 1~2 个交换动作
            n_swaps = 1 if self.rng.random() < 0.85 else 2
            swaps = []
            for _ in range(n_swaps):
                mv = self._propose_swap(selected)
                if mv is not None:
                    swaps.append(mv)

            if not swaps:
                continue

            cand_idx = np.flatnonzero(selected)
            cand_eval = self._evaluate(cand_idx)

            temp = max(1e-9, self.init_temp * (self.cooling ** it))
            accept = False

            if cand_eval["score"] <= cur_eval["score"]:
                accept = True
            else:
                delta = cand_eval["score"] - cur_eval["score"]
                # 防止极大 delta 导致无意义 exp 计算
                if delta < 50.0:
                    accept = self.rng.random() < math.exp(-delta / temp)

            if accept:
                current_idx = cand_idx
                cur_eval = cand_eval

                if cand_eval["score"] + 1e-12 < best_eval["score"]:
                    best_idx = current_idx.copy()
                    best_eval = dict(cand_eval)
                    no_improve = 0
                else:
                    no_improve += 1
            else:
                # 回滚交换
                for out_i, in_i in reversed(swaps):
                    selected[out_i] = True
                    selected[in_i] = False
                no_improve += 1

            if (it + 1) % 25 == 0:
                run_history.append(
                    {
                        "iter": it + 1,
                        "score": cur_eval["score"],
                        "best_score": best_eval["score"],
                        "bimodal_count": cur_eval["bimodal_count"],
                        "extreme_hard_count": cur_eval["extreme_hard_count"],
                        "stat_violation_count": cur_eval["stat_violation_count"],
                        "avg_stat_diff": cur_eval["avg_stat_diff"],
                        "shape_penalty": cur_eval["shape_penalty"],
                        "tail_penalty": cur_eval["tail_penalty"],
                        "outlier_penalty": cur_eval["outlier_penalty"],
                        "extreme_penalty": cur_eval["extreme_penalty"],
                        "normality_penalty": cur_eval["normality_penalty"],
                    }
                )

            if no_improve >= self.early_stop:
                break

        return best_idx, best_eval, run_history

    def _build_column_report(self, best_idx: np.ndarray) -> pd.DataFrame:
        X_sub = self.X_all[best_idx, :]

        sub_q1, sub_med, sub_q3, sub_std = self._compute_stats(X_sub)
        sub_shape_all = np.nanpercentile(X_sub, self.shape_q, axis=0)
        shape_dist = np.mean(
            np.abs(sub_shape_all - self.ref_shape_all) / (self.scale_all + 1e-6), axis=0
        )
        shape_dist = np.nan_to_num(shape_dist, nan=1e3, posinf=1e3, neginf=1e3)

        sub_bimodal = np.array([self._is_bimodal_cfg(X_sub[:, j]) for j in self.all_pos], dtype=bool)

        med_diff = np.abs(sub_med - self.ref_med_all)
        q1_diff = np.abs(sub_q1 - self.ref_q1_all)
        q3_diff = np.abs(sub_q3 - self.ref_q3_all)
        std_diff = np.abs(sub_std - self.ref_std_all)

        med_diff = np.nan_to_num(med_diff, nan=1e3, posinf=1e3, neginf=1e3)
        q1_diff = np.nan_to_num(q1_diff, nan=1e3, posinf=1e3, neginf=1e3)
        q3_diff = np.nan_to_num(q3_diff, nan=1e3, posinf=1e3, neginf=1e3)
        std_diff = np.nan_to_num(std_diff, nan=1e3, posinf=1e3, neginf=1e3)

        # 额外：全 93 列尾部/异常率报告
        ref_p01_all = np.nanpercentile(self.X_ref_all, 1, axis=0)
        ref_p99_all = np.nanpercentile(self.X_ref_all, 99, axis=0)
        sub_p01_all = np.nanpercentile(X_sub, 1, axis=0)
        sub_p99_all = np.nanpercentile(X_sub, 99, axis=0)

        tail_ref_all = ref_p99_all - ref_p01_all
        tail_sub_all = sub_p99_all - sub_p01_all
        iqr_all = self.ref_q3_all - self.ref_q1_all
        tail_scale_all = np.where(
            tail_ref_all > 1e-6,
            tail_ref_all,
            np.where(iqr_all > 1e-6, iqr_all, np.where(self.ref_std_all > 1e-6, self.ref_std_all, 1.0)),
        )
        tail_diff_norm = np.abs(tail_sub_all - tail_ref_all) / (tail_scale_all + 1e-6)

        iqr_safe_all = np.where(
            iqr_all > 1e-6,
            iqr_all,
            np.where(self.ref_std_all > 1e-6, self.ref_std_all, 1.0),
        )
        low15_all = self.ref_q1_all - 1.5 * iqr_safe_all
        up15_all = self.ref_q3_all + 1.5 * iqr_safe_all
        low30_all = self.ref_q1_all - 3.0 * iqr_safe_all
        up30_all = self.ref_q3_all + 3.0 * iqr_safe_all

        ref_out15 = np.nanmean((self.X_ref_all < low15_all) | (self.X_ref_all > up15_all), axis=0)
        sub_out15 = np.nanmean((X_sub < low15_all) | (X_sub > up15_all), axis=0)
        ref_out30 = np.nanmean((self.X_ref_all < low30_all) | (self.X_ref_all > up30_all), axis=0)
        sub_out30 = np.nanmean((X_sub < low30_all) | (X_sub > up30_all), axis=0)

        tail_diff_norm = np.nan_to_num(tail_diff_norm, nan=1e3, posinf=1e3, neginf=1e3)
        ref_out15 = np.nan_to_num(ref_out15, nan=0.0, posinf=0.0, neginf=0.0)
        sub_out15 = np.nan_to_num(sub_out15, nan=0.0, posinf=0.0, neginf=0.0)
        ref_out30 = np.nan_to_num(ref_out30, nan=0.0, posinf=0.0, neginf=0.0)
        sub_out30 = np.nan_to_num(sub_out30, nan=0.0, posinf=0.0, neginf=0.0)

        out_rate15_diff = np.abs(sub_out15 - ref_out15)
        out_rate30_excess = np.maximum(0.0, sub_out30 - (ref_out30 + 0.01))

        is_fc = np.array([str(c).startswith("FC") for c in self.all_cols], dtype=bool)
        target_set = set(self.target_cols)
        is_target = np.array([c in target_set for c in self.all_cols], dtype=bool)

        # 参考双峰标记（仅 target 列有该特殊规则，非 target 列置 False）
        target_bi_map = {c: bool(b) for c, b in zip(self.target_cols, self.t_bimodal)}
        special_rule = np.array([target_bi_map.get(c, False) for c in self.all_cols], dtype=bool)

        # FC 正态指标
        fc_skew = np.full(len(self.all_cols), np.nan, dtype=float)
        fc_kurt = np.full(len(self.all_cols), np.nan, dtype=float)
        for p in self.fc_pos_all:
            col = X_sub[:, p]
            y = col[np.isfinite(col)]
            if y.size >= 8:
                fc_skew[p] = float(skew(y, bias=False))
                fc_kurt[p] = float(kurtosis(y, fisher=False, bias=False))

        fc_norm_pen = np.full(len(self.all_cols), np.nan, dtype=float)
        for p in self.fc_pos_all:
            skv = fc_skew[p]
            kuv = fc_kurt[p]
            if np.isfinite(skv) and np.isfinite(kuv):
                fc_norm_pen[p] = max(0.0, abs(skv) - self.normal_skew_thr) + max(
                    0.0, abs(kuv - 3.0) - self.normal_kurt_thr
                )

        max_diff = np.maximum.reduce([med_diff, q1_diff, q3_diff, std_diff])
        above_thr = max_diff > self.stat_threshold

        report = pd.DataFrame(
            {
                "column": self.all_cols,
                "is_target": is_target,
                "is_fc": is_fc,
                "ref_bimodal": self.ref_bimodal_all,
                "target_special_rule_ref_bimodal": special_rule,
                "subset_bimodal": sub_bimodal,
                "shape_distance": shape_dist,
                "median_diff": med_diff,
                "q1_diff": q1_diff,
                "q3_diff": q3_diff,
                "std_diff": std_diff,
                "max_diff": max_diff,
                "above_stat_threshold": above_thr,
                "tail_diff_norm": tail_diff_norm,
                "out_rate15_ref": ref_out15,
                "out_rate15_sub": sub_out15,
                "out_rate15_diff": out_rate15_diff,
                "out_rate30_ref": ref_out30,
                "out_rate30_sub": sub_out30,
                "out_rate30_excess": out_rate30_excess,
                "fc_skew": fc_skew,
                "fc_kurtosis": fc_kurt,
                "fc_normality_penalty": fc_norm_pen,
            }
        )

        return report

    def fit(self, df_C: pd.DataFrame, df_ref: pd.DataFrame):
        self._prepare(df_C=df_C, df_ref=df_ref)

        best_global_idx = None
        best_global_eval = None
        history_rows = []

        for run in range(1, self.n_starts + 1):
            init_idx = self._sample_initial_indices()
            best_idx, best_eval, run_hist = self._optimize_once(init_idx)

            for row in run_hist:
                row["run"] = run
                history_rows.append(row)

            if (best_global_eval is None) or (best_eval["score"] < best_global_eval["score"]):
                best_global_idx = best_idx
                best_global_eval = best_eval

        final_eval = self._evaluate(best_global_idx, detail=True)

        subset_df = self.df_c.iloc[best_global_idx].copy()
        subset_df = subset_df.sort_values("__orig_index__")
        subset_df.index = subset_df["__orig_index__"].to_numpy()
        subset_df = subset_df.drop(columns=["__orig_index__"])

        col_report = self._build_column_report(best_global_idx)
        history_df = pd.DataFrame(history_rows)

        bp, pp, mp = final_eval["judge_props"]
        summary = {
            "target_mode": self.target_mode,
            "n_target_columns": len(self.target_cols),
            "n_fc_columns": len(self.fc_cols),
            "subset_size": int(len(best_global_idx)),
            "size_ok": bool(final_eval["size_ok"]),
            "judge_ok": bool(final_eval["judge_ok"]),
            "judge_best": float(bp),
            "judge_pass": float(pp),
            "judge_medium": float(mp),
            "all_non_bimodal": bool(final_eval["bimodal_count"] == 0),
            "bimodal_columns_count": int(final_eval["bimodal_count"]),
            "extreme_hard_count": int(final_eval["extreme_hard_count"]),
            "stat_violation_count": int(final_eval["stat_violation_count"]),
            "avg_stat_diff": float(final_eval["avg_stat_diff"]),
            "shape_penalty": float(final_eval["shape_penalty"]),
            "tail_penalty": float(final_eval["tail_penalty"]),
            "outlier_penalty": float(final_eval["outlier_penalty"]),
            "extreme_penalty": float(final_eval["extreme_penalty"]),
            "fc_normality_penalty": float(final_eval["normality_penalty"]),
            "final_score": float(final_eval["score"]),
            "fixed_sample_size": int(self.fixed_n),
            "fixed_counts": {
                "BEST": int(self.fixed_counts[0]),
                "PASS": int(self.fixed_counts[1]),
                "MEDIUM": int(self.fixed_counts[2]),
            },
        }

        return subset_df, summary, col_report, history_df


# ---------------------------
# Example usage
# ---------------------------
if __name__ == "__main__":
    df_C = load_table(r"C:\Users\HP\Desktop\code\C.csv")

    # 规范化并计数
    hash_norm = df_C["#"].fillna("<NA>").astype(str).str.strip()
    counts = hash_norm.value_counts()
    counts_df = (
        counts.rename_axis("#")
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    # 保留 count > 320 的类别并取第一个
    classes_over_320 = counts[counts > 320].index.tolist()
    if classes_over_320:
        first_val = classes_over_320[0]
        C_data = df_C[hash_norm == first_val].copy()
        print(f"Selected first class: {first_val}, n={len(C_data)}")
    else:
        C_data = df_C.iloc[0:0].copy()
        print("No class with count > 320 found; C_data is empty.")

    df_C30 = load_table(r"C:\Users\HP\Desktop\code\C30.csv")
    data_ref = df_C30[df_C30["#"] == "0.0XM C3.0_CGS"].copy()
    if data_ref.empty:
        raise ValueError("df_C30 参考组为空，请检查过滤条件。")

    optimizer = FastSubsetOptimizer(
        target_mode="all_93",
        stat_threshold=3.0,
        size_min=320,
        size_max=340,
        target_size=330,
        judge_tolerance=0.015,
        n_starts=5,
        max_iter=900,
        early_stop=220,
        random_seed=42,
        w_shape=1.2,
        w_stat=1.0,
        w_norm=0.6,
        w_stat_violation=0.2,
        w_tail=1.2,
        w_outlier=2.5,
        w_extreme=8.0,
        hard_extreme_margin=0.02,
        bimodal_sep_ratio=0.08,
        bimodal_valley_ratio=0.78,
        bimodal_min_side_mass=0.12,
    )

    subset_df, summary, col_report, history_df = optimizer.fit(df_C=C_data, df_ref=data_ref)

    print("Summary:")
    for k, v in summary.items():
        print(f"  {k}: {v}")

    subset_df.to_excel(r"C:\Users\HP\Desktop\code\optimal_subset.xlsx", index=False)
    col_report.to_excel(r"C:\Users\HP\Desktop\code\col_report.xlsx", index=False)
    history_df.to_excel(r"C:\Users\HP\Desktop\code\search_history.xlsx", index=False)

Selected first class: 5C-C5, n=1727
Summary:
  target_mode: all_93
  n_target_columns: 93
  n_fc_columns: 22
  subset_size: 330
  size_ok: True
  judge_ok: True
  judge_best: 0.47575757575757577
  judge_pass: 0.44545454545454544
  judge_medium: 0.07878787878787878
  all_non_bimodal: True
  bimodal_columns_count: 0
  extreme_hard_count: 3
  stat_violation_count: 7
  avg_stat_diff: 0.9353906836908972
  shape_penalty: 0.5455625490935504
  tail_penalty: 0.21051041268594434
  outlier_penalty: 0.040233470131172774
  extreme_penalty: 0.003816310305794447
  fc_normality_penalty: 0.05049693584442139
  final_score: 600002.0191443205
  fixed_sample_size: 330
  fixed_counts: {'BEST': 157, 'PASS': 147, 'MEDIUM': 26}


In [24]:
import math
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew

warnings.filterwarnings("ignore", category=RuntimeWarning)


def load_table(path: str) -> pd.DataFrame:
    """
    通用读表函数：
    - .csv 用 read_csv（low_memory=False，避免 dtype 分块推断警告）
    - 其他后缀默认按 Excel 读取
    """
    p = str(path).lower()
    if p.endswith(".csv"):
        return pd.read_csv(path, low_memory=False)
    return pd.read_excel(path, engine="openpyxl")


class FastSubsetOptimizer:
    """
    从 df_C 中抽取 size_min~size_max 的样本子集，使其尽量接近 df_ref（参考组）。

    约束与目标：
    1) Judge 比例满足约束（BEST/PASS/MEDIUM）
    2) 所有 93 列尽量避免双峰
    3) 统计量（q1/median/q3/std）接近参考
    4) 形态（分位点曲线）接近参考
    5) 尾部宽度、异常值比例接近参考
    6) FC 列尽量接近正态分布
    7) 可选：prioritize_fc=True 时，对 FC 列施加更严格的双峰与正态要求
    """

    def __init__(
        self,
        target_mode: str = "all_93",          # "all_93" 或 "fc_only"
        size_min: int = 320,
        size_max: int = 340,
        target_size: int = 330,
        judge_tolerance: float = 0.015,
        stat_threshold: float = 3.0,
        n_starts: int = 5,
        max_iter: int = 900,
        early_stop: int = 220,
        init_temp: float = 0.08,
        cooling: float = 0.997,
        normal_skew_thr: float = 0.5,
        normal_kurt_thr: float = 1.5,
        random_seed: int = 42,
        # 软约束权重
        w_shape: float = 1.2,
        w_stat: float = 1.0,
        w_norm: float = 0.6,
        w_stat_violation: float = 0.2,
        w_tail: float = 1.2,
        w_outlier: float = 2.5,
        w_extreme: float = 8.0,
        # 极端值硬约束余量（3IQR 外比例超出参考多少算硬违规）
        hard_extreme_margin: float = 0.02,
        # 常规双峰判定参数
        bimodal_min_n: int = 30,
        bimodal_peak_height_ratio: float = 0.10,
        bimodal_sep_ratio: float = 0.08,
        bimodal_valley_ratio: float = 0.78,
        bimodal_min_side_mass: float = 0.12,
        # ===== FC 优先模式参数（可选）=====
        prioritize_fc: bool = False,
        fc_weight_mult: float = 2.0,          # FC 列在统计/形态/尾部等项中的加权倍数
        # FC 双峰更严格参数（更容易判为双峰，能抓“葫芦型”）
        fc_bimodal_peak_height_ratio: float = 0.06,
        fc_bimodal_sep_ratio: float = 0.05,
        fc_bimodal_valley_ratio: float = 0.92,
        fc_bimodal_min_side_mass: float = 0.08,
        # FC 正态更严格阈值
        fc_normal_skew_thr: float = 0.35,
        fc_normal_kurt_thr: float = 1.0,
        # FC 正态惩罚增强倍数（prioritize_fc=True 时生效）
        fc_norm_weight_mult: float = 2.0,
        # FC 正态硬违规阈值缓冲（超过阈值 + margin 算硬违规）
        fc_norm_hard_margin: float = 0.25,
    ):
        # 基础配置
        self.target_mode = target_mode
        self.size_min = size_min
        self.size_max = size_max
        self.target_size = target_size
        self.judge_tolerance = judge_tolerance
        self.stat_threshold = stat_threshold

        # 搜索配置
        self.n_starts = n_starts
        self.max_iter = max_iter
        self.early_stop = early_stop
        self.init_temp = init_temp
        self.cooling = cooling

        # 正态阈值
        self.normal_skew_thr = normal_skew_thr
        self.normal_kurt_thr = normal_kurt_thr

        # 软约束权重
        self.w_shape = w_shape
        self.w_stat = w_stat
        self.w_norm = w_norm
        self.w_stat_violation = w_stat_violation
        self.w_tail = w_tail
        self.w_outlier = w_outlier
        self.w_extreme = w_extreme

        # 硬约束参数
        self.hard_extreme_margin = hard_extreme_margin

        # 常规双峰参数
        self.bimodal_min_n = bimodal_min_n
        self.bimodal_peak_height_ratio = bimodal_peak_height_ratio
        self.bimodal_sep_ratio = bimodal_sep_ratio
        self.bimodal_valley_ratio = bimodal_valley_ratio
        self.bimodal_min_side_mass = bimodal_min_side_mass

        # FC 优先参数
        self.prioritize_fc = prioritize_fc
        self.fc_weight_mult = fc_weight_mult
        self.fc_bimodal_peak_height_ratio = fc_bimodal_peak_height_ratio
        self.fc_bimodal_sep_ratio = fc_bimodal_sep_ratio
        self.fc_bimodal_valley_ratio = fc_bimodal_valley_ratio
        self.fc_bimodal_min_side_mass = fc_bimodal_min_side_mass
        self.fc_normal_skew_thr = fc_normal_skew_thr
        self.fc_normal_kurt_thr = fc_normal_kurt_thr
        self.fc_norm_weight_mult = fc_norm_weight_mult
        self.fc_norm_hard_margin = fc_norm_hard_margin

        # 随机数生成器（统一随机入口，便于复现）
        self.rng = np.random.default_rng(random_seed)

        # 分位点网格：用于形态相似度计算
        self.shape_q = np.array([5, 15, 25, 35, 50, 65, 75, 85, 95], dtype=np.float64)

        # ===== 运行时在 _prepare 中填充 =====
        self.df_c = None
        self.df_ref = None

        self.all_cols = None
        self.fc_cols = None
        self.target_cols = None

        self.X_all = None
        self.X_ref_all = None

        self.n_rows = 0
        self.judge_code = None
        self.pool = None  # {0: BEST索引数组, 1: PASS索引数组, 2: MEDIUM索引数组}

        self.all_pos = None
        self.target_pos = None
        self.fc_pos_all = None

        # target 列的权重（FC 优先时会加权）
        self.target_is_fc = None
        self.target_col_weight = None

        # target 列参考统计
        self.t_q1 = None
        self.t_med = None
        self.t_q3 = None
        self.t_std = None
        self.t_shape = None
        self.t_scale = None
        self.t_bimodal = None

        # target 列参考尾部/异常值统计
        self.ref_p01 = None
        self.ref_p99 = None
        self.ref_iqr = None
        self.tail_scale = None
        self.ref_low_15 = None
        self.ref_up_15 = None
        self.ref_low_30 = None
        self.ref_up_30 = None
        self.ref_out_rate_15 = None
        self.ref_out_rate_30 = None

        # all 93 列参考统计（报表用）
        self.ref_q1_all = None
        self.ref_med_all = None
        self.ref_q3_all = None
        self.ref_std_all = None
        self.ref_shape_all = None
        self.scale_all = None
        self.ref_bimodal_all = None

        # 固定可行样本量与固定 Judge 计数（用于快速同类交换）
        self.fixed_n = None
        self.fixed_counts = None  # {0: BEST数, 1: PASS数, 2: MEDIUM数}

    @staticmethod
    def _extract_93_columns(df: pd.DataFrame) -> List[str]:
        """
        从 EFL 开始截取连续 93 列，作为目标列集合。
        """
        if "EFL" not in df.columns:
            raise ValueError("未找到列 EFL，无法定位目标 93 列。")
        start = df.columns.get_loc("EFL")
        cols = list(df.columns[start:start + 93])
        if len(cols) != 93:
            raise ValueError(f"从 EFL 开始不足 93 列，实际只有 {len(cols)} 列。")
        return cols

    @staticmethod
    def _normalize_judge(s: pd.Series) -> pd.Series:
        """
        Judge 标准化：
        - 去空格
        - 转大写
        - 修正常见拼写 BSET -> BEST
        """
        x = s.astype(str).str.strip().str.upper()
        x = x.replace({"BSET": "BEST"})
        return x

    @staticmethod
    def _compute_stats(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        统一计算 q1/median/q3/std（按列）。
        """
        q = np.nanpercentile(X, [25, 50, 75], axis=0)
        q1, med, q3 = q[0], q[1], q[2]
        std = np.nanstd(X, axis=0, ddof=1)
        std = np.nan_to_num(std, nan=0.0, posinf=0.0, neginf=0.0)
        return q1, med, q3, std

    @staticmethod
    def _is_bimodal(
        x: np.ndarray,
        min_n: int = 30,
        peak_height_ratio: float = 0.10,
        sep_ratio: float = 0.08,
        valley_ratio: float = 0.78,
        min_side_mass: float = 0.12,
    ) -> bool:
        """
        双峰判定（直方图 + 平滑）：
        1) 至少有两个显著峰
        2) 两峰有足够分离
        3) 峰间谷足够低（valley_ratio 越大越“严格敏感”）
        4) 谷两侧样本质量不能太小，避免噪声假峰
        """
        y = np.asarray(x, dtype=np.float64)
        y = y[np.isfinite(y)]
        n = y.size
        if n < min_n:
            return False

        y_min = float(np.min(y))
        y_max = float(np.max(y))
        if (not np.isfinite(y_min)) or (not np.isfinite(y_max)) or (y_max <= y_min):
            return False

        bins = int(np.clip(np.sqrt(n), 12, 48))
        hist, edges = np.histogram(y, bins=bins)
        if hist.max() <= 0:
            return False

        # 三点卷积做轻量平滑
        smooth = np.convolve(hist.astype(np.float64), np.array([1.0, 2.0, 1.0]), mode="same")
        thr = peak_height_ratio * float(np.max(smooth))

        left = smooth[:-2]
        mid = smooth[1:-1]
        right = smooth[2:]
        peak_idx = np.where((mid > left) & (mid >= right) & (mid >= thr))[0] + 1
        if peak_idx.size < 2:
            return False

        # 取最高两峰
        top2 = peak_idx[np.argsort(smooth[peak_idx])[-2:]]
        p1, p2 = int(min(top2)), int(max(top2))

        centers = 0.5 * (edges[:-1] + edges[1:])
        sep = abs(centers[p1] - centers[p2]) / (y_max - y_min + 1e-12)
        if sep < sep_ratio:
            return False

        # 峰间谷深
        valley_pos = p1 + int(np.argmin(smooth[p1:p2 + 1]))
        valley = float(smooth[valley_pos])
        min_peak = float(min(smooth[p1], smooth[p2])) + 1e-12
        if valley > valley_ratio * min_peak:
            return False

        # 谷左右质量
        left_mass = float(hist[:valley_pos + 1].sum()) / max(1, n)
        right_mass = float(hist[valley_pos + 1:].sum()) / max(1, n)
        if min(left_mass, right_mass) < min_side_mass:
            return False

        return True

    def _is_bimodal_cfg(self, x: np.ndarray) -> bool:
        """
        使用常规参数的双峰检测。
        """
        return self._is_bimodal(
            x=x,
            min_n=self.bimodal_min_n,
            peak_height_ratio=self.bimodal_peak_height_ratio,
            sep_ratio=self.bimodal_sep_ratio,
            valley_ratio=self.bimodal_valley_ratio,
            min_side_mass=self.bimodal_min_side_mass,
        )

    def _is_bimodal_fc(self, x: np.ndarray) -> bool:
        """
        FC 严格双峰检测：
        - prioritize_fc=False 时，退化为常规检测
        - prioritize_fc=True 时，用更敏感参数（更容易识别“葫芦型”）
        """
        if not self.prioritize_fc:
            return self._is_bimodal_cfg(x)

        return self._is_bimodal(
            x=x,
            min_n=self.bimodal_min_n,
            peak_height_ratio=self.fc_bimodal_peak_height_ratio,
            sep_ratio=self.fc_bimodal_sep_ratio,
            valley_ratio=self.fc_bimodal_valley_ratio,
            min_side_mass=self.fc_bimodal_min_side_mass,
        )

    def _prepare(self, df_C: pd.DataFrame, df_ref: pd.DataFrame) -> None:
        """
        预处理阶段（只做一次）：
        1) 定位 93 列/FC 列
        2) 标准化 Judge
        3) 全量转 numpy 提速
        4) 计算参考统计（目标统计、形态、尾部、异常率）
        5) 找到固定可行样本量和固定 Judge 计数
        """
        self.df_c = df_C.copy()
        self.df_ref = df_ref.copy()

        # 1) 目标列检查
        self.all_cols = self._extract_93_columns(self.df_ref)
        missing = [c for c in self.all_cols if c not in self.df_c.columns]
        if missing:
            raise ValueError(f"df_C 缺少目标列: {missing[:5]} ...")

        self.fc_cols = [c for c in self.all_cols if str(c).startswith("FC")]
        if self.target_mode == "all_93":
            self.target_cols = list(self.all_cols)
        elif self.target_mode == "fc_only":
            if not self.fc_cols:
                raise ValueError("target_mode='fc_only' 但未找到 FC 列。")
            self.target_cols = list(self.fc_cols)
        else:
            raise ValueError("target_mode 只能是 'all_93' 或 'fc_only'。")

        # 2) Judge 标准化 + 仅保留合法类别
        if "Judge" not in self.df_c.columns:
            raise ValueError("df_C 缺少 Judge 列。")
        self.df_c["Judge"] = self._normalize_judge(self.df_c["Judge"])

        valid_mask = self.df_c["Judge"].isin(["BEST", "PASS", "MEDIUM"])
        self.df_c = self.df_c.loc[valid_mask].copy()
        if self.df_c.empty:
            raise ValueError("df_C 在过滤 Judge 后为空。")

        # 保存原始索引，后续输出按原顺序还原
        self.df_c = self.df_c.reset_index(drop=False).rename(columns={"index": "__orig_index__"})

        # 3) 一次性转 numpy，后续搜索全走 numpy（性能关键）
        self.X_all = (
            self.df_c[self.all_cols]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float32)
        )
        self.X_ref_all = (
            self.df_ref[self.all_cols]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float32)
        )

        self.n_rows = self.X_all.shape[0]
        self.all_pos = np.arange(len(self.all_cols), dtype=np.int32)
        col_to_pos = {c: i for i, c in enumerate(self.all_cols)}
        self.target_pos = np.array([col_to_pos[c] for c in self.target_cols], dtype=np.int32)
        self.fc_pos_all = np.array([col_to_pos[c] for c in self.fc_cols], dtype=np.int32)

        # target 列内哪些是 FC，用于加权
        self.target_is_fc = np.isin(self.target_pos, self.fc_pos_all)
        if self.prioritize_fc:
            self.target_col_weight = np.where(self.target_is_fc, self.fc_weight_mult, 1.0).astype(np.float64)
        else:
            self.target_col_weight = np.ones(self.target_pos.shape[0], dtype=np.float64)

        # 4) Judge 编码（便于快速计数）
        code_map = {"BEST": 0, "PASS": 1, "MEDIUM": 2}
        self.judge_code = self.df_c["Judge"].map(code_map).to_numpy(dtype=np.int8)
        self.pool = {j: np.where(self.judge_code == j)[0] for j in (0, 1, 2)}

        # 5) 目标列参考统计
        X_ref_target = self.X_ref_all[:, self.target_pos]
        self.t_q1, self.t_med, self.t_q3, self.t_std = self._compute_stats(X_ref_target)

        # 形态基线（多分位点）
        self.t_shape = np.nanpercentile(X_ref_target, self.shape_q, axis=0)

        # 用 IQR/std 构造尺度，避免不同量纲列互相压制
        t_iqr = self.t_q3 - self.t_q1
        self.t_scale = np.where(t_iqr > 1e-6, t_iqr, np.where(self.t_std > 1e-6, self.t_std, 1.0))

        # 参考中目标列的双峰标记（用于 shape 豁免）
        self.t_bimodal = np.array(
            [self._is_bimodal_cfg(X_ref_target[:, i]) for i in range(X_ref_target.shape[1])],
            dtype=bool,
        )

        # 尾部宽度基线：p99 - p1
        self.ref_p01 = np.nanpercentile(X_ref_target, 1, axis=0)
        self.ref_p99 = np.nanpercentile(X_ref_target, 99, axis=0)
        self.ref_p01 = np.nan_to_num(self.ref_p01, nan=0.0, posinf=0.0, neginf=0.0)
        self.ref_p99 = np.nan_to_num(self.ref_p99, nan=0.0, posinf=0.0, neginf=0.0)

        self.ref_iqr = self.t_q3 - self.t_q1
        self.ref_iqr = np.nan_to_num(self.ref_iqr, nan=0.0, posinf=0.0, neginf=0.0)

        tail_ref = self.ref_p99 - self.ref_p01
        self.tail_scale = np.where(
            tail_ref > 1e-6,
            tail_ref,
            np.where(self.ref_iqr > 1e-6, self.ref_iqr, np.where(self.t_std > 1e-6, self.t_std, 1.0)),
        )

        # 异常值上下界（基于参考分布）
        iqr_safe = np.where(
            self.ref_iqr > 1e-6,
            self.ref_iqr,
            np.where(self.t_std > 1e-6, self.t_std, 1.0),
        )
        self.ref_low_15 = self.t_q1 - 1.5 * iqr_safe
        self.ref_up_15 = self.t_q3 + 1.5 * iqr_safe
        self.ref_low_30 = self.t_q1 - 3.0 * iqr_safe
        self.ref_up_30 = self.t_q3 + 3.0 * iqr_safe

        ref_out_15 = (X_ref_target < self.ref_low_15) | (X_ref_target > self.ref_up_15)
        ref_out_30 = (X_ref_target < self.ref_low_30) | (X_ref_target > self.ref_up_30)
        self.ref_out_rate_15 = np.nanmean(ref_out_15, axis=0)
        self.ref_out_rate_30 = np.nanmean(ref_out_30, axis=0)
        self.ref_out_rate_15 = np.nan_to_num(self.ref_out_rate_15, nan=0.0, posinf=0.0, neginf=0.0)
        self.ref_out_rate_30 = np.nan_to_num(self.ref_out_rate_30, nan=0.0, posinf=0.0, neginf=0.0)

        # 6) all 93 列参考统计（报表用）
        self.ref_q1_all, self.ref_med_all, self.ref_q3_all, self.ref_std_all = self._compute_stats(self.X_ref_all)
        self.ref_shape_all = np.nanpercentile(self.X_ref_all, self.shape_q, axis=0)

        iqr_all = self.ref_q3_all - self.ref_q1_all
        self.scale_all = np.where(
            iqr_all > 1e-6,
            iqr_all,
            np.where(self.ref_std_all > 1e-6, self.ref_std_all, 1.0),
        )

        self.ref_bimodal_all = np.array(
            [self._is_bimodal_cfg(self.X_ref_all[:, j]) for j in self.all_pos],
            dtype=bool,
        )

        # 7) 固定可行样本量与固定计数（保证后续同类交换始终合法）
        self.fixed_n, self.fixed_counts = self._find_feasible_size_and_counts()

    def _find_feasible_size_and_counts(self) -> Tuple[int, Dict[int, int]]:
        """
        在 size_min~size_max 内寻找可行 n，并求 BEST/PASS/MEDIUM 的固定计数。
        优先：离 target_size 最近，且比例偏差最小。
        """
        avail_best = len(self.pool[0])
        avail_pass = len(self.pool[1])
        avail_medium = len(self.pool[2])

        size_candidates = list(range(self.size_min, self.size_max + 1))
        size_candidates.sort(key=lambda n: (abs(n - self.target_size), n))

        best_solution = None  # (size_distance, ratio_deviation, n, counts_dict)

        for n in size_candidates:
            # BEST 与 PASS 按硬范围
            b_lo, b_hi = math.ceil(0.45 * n), math.floor(0.50 * n)
            p_lo, p_hi = math.ceil(0.42 * n), math.floor(0.47 * n)
            # MEDIUM 允许 tolerance
            m_lo = math.ceil((0.08 - self.judge_tolerance) * n)
            m_hi = math.floor((0.08 + self.judge_tolerance) * n)
            m_lo = max(0, m_lo)

            if b_lo > b_hi or p_lo > p_hi or m_lo > m_hi:
                continue

            local_best = None
            for nb in range(b_lo, b_hi + 1):
                if nb > avail_best:
                    continue
                for npass in range(p_lo, p_hi + 1):
                    if npass > avail_pass:
                        continue

                    nmed = n - nb - npass
                    if nmed < m_lo or nmed > m_hi:
                        continue
                    if nmed > avail_medium:
                        continue

                    dev = abs(nb / n - 0.475) + abs(npass / n - 0.445) + abs(nmed / n - 0.08)
                    cand = (dev, {0: nb, 1: npass, 2: nmed})
                    if (local_best is None) or (cand[0] < local_best[0]):
                        local_best = cand

            if local_best is not None:
                size_distance = abs(n - self.target_size)
                solution = (size_distance, local_best[0], n, local_best[1])
                if (best_solution is None) or (solution < best_solution):
                    best_solution = solution
                # size_candidates 已按距离排序，首个可行解即可提前退出
                break

        if best_solution is None:
            raise ValueError("在目标范围内找不到满足 Judge 约束的可行解，请检查 df_C Judge 分布。")

        return best_solution[2], best_solution[3]

    def _sample_initial_indices(self) -> np.ndarray:
        """
        按固定计数从三类中随机采样，形成一个可行初始解。
        """
        nb = self.fixed_counts[0]
        npa = self.fixed_counts[1]
        nmed = self.fixed_counts[2]

        idx_b = self.rng.choice(self.pool[0], size=nb, replace=False)
        idx_p = self.rng.choice(self.pool[1], size=npa, replace=False)
        idx_m = self.rng.choice(self.pool[2], size=nmed, replace=False)

        idx = np.concatenate([idx_b, idx_p, idx_m])
        self.rng.shuffle(idx)
        return idx

    def _judge_ok(self, cnt: np.ndarray, n: int) -> Tuple[bool, Tuple[float, float, float], float]:
        """
        检查 Judge 比例是否满足约束，并返回偏离度（用于硬惩罚强度）。
        """
        bp = cnt[0] / n
        pp = cnt[1] / n
        mp = cnt[2] / n

        ok = (
            (0.45 <= bp <= 0.50)
            and (0.42 <= pp <= 0.47)
            and (abs(mp - 0.08) <= self.judge_tolerance)
        )

        dev = 0.0
        dev += max(0.0, 0.45 - bp, bp - 0.50)
        dev += max(0.0, 0.42 - pp, pp - 0.47)
        dev += max(0.0, abs(mp - 0.08) - self.judge_tolerance)

        return ok, (bp, pp, mp), dev

    def _evaluate(self, idx: np.ndarray, detail: bool = False) -> Dict:
        """
        目标函数评估：
        - 软项：统计、形态、尾部、异常值、FC 正态
        - 硬项：size、judge、双峰、极端尾部、（可选）FC 严格双峰、FC 严格正态
        """
        n = idx.size

        # ---------- 目标列基础统计 ----------
        X_t = self.X_all[np.ix_(idx, self.target_pos)]  # [n, n_target]
        q = np.nanpercentile(X_t, [25, 50, 75], axis=0)
        q1, med, q3 = q[0], q[1], q[2]
        std = np.nanstd(X_t, axis=0, ddof=1)

        d_med = np.abs(med - self.t_med)
        d_q1 = np.abs(q1 - self.t_q1)
        d_q3 = np.abs(q3 - self.t_q3)
        d_std = np.abs(std - self.t_std)

        # 数值防护
        d_med = np.nan_to_num(d_med, nan=1e3, posinf=1e3, neginf=1e3)
        d_q1 = np.nan_to_num(d_q1, nan=1e3, posinf=1e3, neginf=1e3)
        d_q3 = np.nan_to_num(d_q3, nan=1e3, posinf=1e3, neginf=1e3)
        d_std = np.nan_to_num(d_std, nan=1e3, posinf=1e3, neginf=1e3)

        d_stack = np.vstack([d_med, d_q1, d_q3, d_std])
        d_max = np.max(d_stack, axis=0)
        stat_violation = d_max > self.stat_threshold

        # FC 优先时，FC 列加权更高
        if (self.target_col_weight is None) or (self.target_col_weight.shape[0] != d_med.shape[0]):
            col_w = np.ones(d_med.shape[0], dtype=np.float64)
        else:
            col_w = self.target_col_weight

        stat_penalty_each = np.mean(d_stack, axis=0)
        stat_penalty = float(np.average(stat_penalty_each, weights=col_w))
        stat_violation_rate = float(np.average(stat_violation.astype(np.float64), weights=col_w))

        # ---------- 形态相似 ----------
        # 对“参考本身双峰”的目标列，不要求形态一致（避免把参考异常形态硬贴过去）
        sub_shape = np.nanpercentile(X_t, self.shape_q, axis=0)
        shape_each = np.mean(np.abs(sub_shape - self.t_shape) / (self.t_scale + 1e-6), axis=0)
        shape_each = np.nan_to_num(shape_each, nan=1e3, posinf=1e3, neginf=1e3)
        shape_each[self.t_bimodal] = 0.0
        shape_penalty = float(np.average(shape_each, weights=col_w))

        # ---------- 尾部宽度惩罚 ----------
        p01 = np.nanpercentile(X_t, 1, axis=0)
        p99 = np.nanpercentile(X_t, 99, axis=0)
        tail_sub = p99 - p01
        tail_ref = self.ref_p99 - self.ref_p01
        tail_penalty_each = np.abs(tail_sub - tail_ref) / (self.tail_scale + 1e-6)
        tail_penalty_each = np.nan_to_num(tail_penalty_each, nan=1e3, posinf=1e3, neginf=1e3)
        tail_penalty = float(np.average(tail_penalty_each, weights=col_w))

        # ---------- 异常值比例惩罚 ----------
        out_15 = (X_t < self.ref_low_15) | (X_t > self.ref_up_15)
        out_30 = (X_t < self.ref_low_30) | (X_t > self.ref_up_30)

        out_rate_15 = np.nanmean(out_15, axis=0)
        out_rate_30 = np.nanmean(out_30, axis=0)
        out_rate_15 = np.nan_to_num(out_rate_15, nan=0.0, posinf=0.0, neginf=0.0)
        out_rate_30 = np.nan_to_num(out_rate_30, nan=0.0, posinf=0.0, neginf=0.0)

        outlier_penalty_each = np.abs(out_rate_15 - self.ref_out_rate_15)
        outlier_penalty_each = np.nan_to_num(outlier_penalty_each, nan=1e3, posinf=1e3, neginf=1e3)
        outlier_penalty = float(np.average(outlier_penalty_each, weights=col_w))

        # 只惩罚“高于参考”的极端比例（减少重尾）
        extreme_excess_each = np.maximum(0.0, out_rate_30 - (self.ref_out_rate_30 + 0.01))
        extreme_penalty = float(np.average(extreme_excess_each, weights=col_w))

        # ---------- FC 正态惩罚 ----------
        normality_penalty = 0.0
        fc_norm_hard_count = 0

        if self.fc_pos_all.size > 0:
            X_fc = self.X_all[np.ix_(idx, self.fc_pos_all)]
            sk = skew(X_fc, axis=0, bias=False, nan_policy="omit")
            ku = kurtosis(X_fc, axis=0, fisher=False, bias=False, nan_policy="omit")
            sk = np.nan_to_num(sk, nan=10.0, posinf=10.0, neginf=10.0)
            ku = np.nan_to_num(ku, nan=10.0, posinf=10.0, neginf=10.0)

            sk_thr = self.fc_normal_skew_thr if self.prioritize_fc else self.normal_skew_thr
            ku_thr = self.fc_normal_kurt_thr if self.prioritize_fc else self.normal_kurt_thr

            norm_each = (
                np.maximum(0.0, np.abs(sk) - sk_thr)
                + np.maximum(0.0, np.abs(ku - 3.0) - ku_thr)
            )
            normality_penalty = float(np.mean(norm_each))

            # FC 优先时，正态项权重进一步增强
            if self.prioritize_fc:
                normality_penalty *= self.fc_norm_weight_mult
                fc_norm_hard_mask = (
                    (np.abs(sk) > (sk_thr + self.fc_norm_hard_margin))
                    | (np.abs(ku - 3.0) > (ku_thr + self.fc_norm_hard_margin))
                )
                fc_norm_hard_count = int(np.sum(fc_norm_hard_mask))

        # ---------- 硬约束 ----------
        size_ok = self.size_min <= n <= self.size_max
        size_dev = 0.0 if size_ok else abs(n - self.target_size) / max(1.0, self.target_size)

        cnt = np.bincount(self.judge_code[idx], minlength=3)
        judge_ok, judge_props, judge_dev = self._judge_ok(cnt, n)

        # 常规双峰检查（全 93 列）
        X_sub_all = self.X_all[idx, :]
        bimodal_flags = np.zeros(len(self.all_cols), dtype=bool)
        for j in self.all_pos:
            bimodal_flags[j] = self._is_bimodal_cfg(X_sub_all[:, j])

        # FC 严格双峰检查
        fc_strict_flags = np.zeros(self.fc_pos_all.size, dtype=bool)
        for k, p in enumerate(self.fc_pos_all):
            fc_strict_flags[k] = self._is_bimodal_fc(X_sub_all[:, p])

        fc_bimodal_count = int(np.sum(fc_strict_flags))

        # FC 优先模式下，把 FC 严格双峰结果并入全局双峰标记
        if self.prioritize_fc and fc_bimodal_count > 0:
            for k, p in enumerate(self.fc_pos_all):
                if fc_strict_flags[k]:
                    bimodal_flags[p] = True

        bimodal_count = int(np.sum(bimodal_flags))

        # 极端尾部硬约束
        extreme_hard_mask = out_rate_30 > (self.ref_out_rate_30 + self.hard_extreme_margin)
        extreme_hard_count = int(np.sum(extreme_hard_mask))

        hard_penalty = 0.0
        hard_count = 0

        if not size_ok:
            hard_penalty += 2e5 * (1.0 + size_dev)
            hard_count += 1

        if not judge_ok:
            hard_penalty += 4e5 * (1.0 + judge_dev)
            hard_count += 1

        if bimodal_count > 0:
            hard_penalty += 3e5 * bimodal_count
            hard_count += 1

        if extreme_hard_count > 0:
            hard_penalty += 2e5 * extreme_hard_count
            hard_count += 1

        # FC 优先时，额外硬惩罚：FC 双峰/FC 强偏离正态
        if self.prioritize_fc and fc_bimodal_count > 0:
            hard_penalty += 8e5 * fc_bimodal_count
            hard_count += 1

        if self.prioritize_fc and fc_norm_hard_count > 0:
            hard_penalty += 3e5 * fc_norm_hard_count
            hard_count += 1

        # 总分（越小越好）
        score = (
            hard_penalty
            + self.w_shape * shape_penalty
            + self.w_stat * stat_penalty
            + self.w_norm * normality_penalty
            + self.w_stat_violation * stat_violation_rate
            + self.w_tail * tail_penalty
            + self.w_outlier * outlier_penalty
            + self.w_extreme * extreme_penalty
        )

        out = {
            "score": float(score),
            "hard_count": int(hard_count),
            "size_ok": bool(size_ok),
            "judge_ok": bool(judge_ok),
            "judge_props": judge_props,
            "bimodal_count": bimodal_count,
            "bimodal_flags": bimodal_flags,
            "fc_bimodal_count": fc_bimodal_count,
            "fc_norm_hard_count": fc_norm_hard_count,
            "extreme_hard_count": extreme_hard_count,
            "stat_violation_count": int(np.sum(stat_violation)),
            "stat_violation_rate": stat_violation_rate,
            "avg_stat_diff": stat_penalty,
            "shape_penalty": shape_penalty,
            "tail_penalty": tail_penalty,
            "outlier_penalty": outlier_penalty,
            "extreme_penalty": extreme_penalty,
            "normality_penalty": normality_penalty,
        }

        if detail:
            out["d_med"] = d_med
            out["d_q1"] = d_q1
            out["d_q3"] = d_q3
            out["d_std"] = d_std
            out["shape_each"] = shape_each
            out["tail_penalty_each"] = tail_penalty_each
            out["out_rate_15"] = out_rate_15
            out["out_rate_30"] = out_rate_30
            out["extreme_excess_each"] = extreme_excess_each
            out["stat_violation_mask"] = stat_violation
            out["fc_strict_flags"] = fc_strict_flags

        return out

    def _propose_swap(self, selected: np.ndarray):
        """
        同类交换（同 Judge 类别内换入换出）：
        - 保证 Judge 比例恒定不变
        - 每次只改变少量样本，便于稳态收敛
        """
        j = int(self.rng.integers(0, 3))
        pool_j = self.pool[j]

        selected_j = pool_j[selected[pool_j]]
        unselected_j = pool_j[~selected[pool_j]]

        if selected_j.size == 0 or unselected_j.size == 0:
            return None

        out_i = int(selected_j[self.rng.integers(0, selected_j.size)])
        in_i = int(unselected_j[self.rng.integers(0, unselected_j.size)])

        selected[out_i] = False
        selected[in_i] = True
        return out_i, in_i

    def _optimize_once(self, init_idx: np.ndarray):
        """
        单次优化：
        - 初始可行解 -> 同类交换邻域搜索
        - 使用模拟退火接受准则
        - 记录 run_history 便于回看收敛过程
        """
        selected = np.zeros(self.n_rows, dtype=bool)
        selected[init_idx] = True

        current_idx = np.flatnonzero(selected)
        cur_eval = self._evaluate(current_idx)

        best_idx = current_idx.copy()
        best_eval = dict(cur_eval)

        no_improve = 0
        run_history = []

        for it in range(self.max_iter):
            # 每轮做 1~2 个交换，兼顾稳定与探索
            n_swaps = 1 if self.rng.random() < 0.85 else 2
            swaps = []
            for _ in range(n_swaps):
                mv = self._propose_swap(selected)
                if mv is not None:
                    swaps.append(mv)

            if not swaps:
                continue

            cand_idx = np.flatnonzero(selected)
            cand_eval = self._evaluate(cand_idx)

            # 温度逐步下降
            temp = max(1e-9, self.init_temp * (self.cooling ** it))
            accept = False

            # 更优解直接接受；更差解按退火概率接受
            if cand_eval["score"] <= cur_eval["score"]:
                accept = True
            else:
                delta = cand_eval["score"] - cur_eval["score"]
                if delta < 50.0:
                    accept = self.rng.random() < math.exp(-delta / temp)

            if accept:
                current_idx = cand_idx
                cur_eval = cand_eval

                if cand_eval["score"] + 1e-12 < best_eval["score"]:
                    best_idx = current_idx.copy()
                    best_eval = dict(cand_eval)
                    no_improve = 0
                else:
                    no_improve += 1
            else:
                # 拒绝则回滚交换
                for out_i, in_i in reversed(swaps):
                    selected[out_i] = True
                    selected[in_i] = False
                no_improve += 1

            # 每 25 步记录一次
            if (it + 1) % 25 == 0:
                run_history.append(
                    {
                        "iter": it + 1,
                        "score": cur_eval["score"],
                        "best_score": best_eval["score"],
                        "bimodal_count": cur_eval["bimodal_count"],
                        "fc_bimodal_count": cur_eval["fc_bimodal_count"],
                        "fc_norm_hard_count": cur_eval["fc_norm_hard_count"],
                        "extreme_hard_count": cur_eval["extreme_hard_count"],
                        "stat_violation_count": cur_eval["stat_violation_count"],
                        "avg_stat_diff": cur_eval["avg_stat_diff"],
                        "shape_penalty": cur_eval["shape_penalty"],
                        "tail_penalty": cur_eval["tail_penalty"],
                        "outlier_penalty": cur_eval["outlier_penalty"],
                        "extreme_penalty": cur_eval["extreme_penalty"],
                        "normality_penalty": cur_eval["normality_penalty"],
                    }
                )

            if no_improve >= self.early_stop:
                break

        return best_idx, best_eval, run_history

    def _build_column_report(self, best_idx: np.ndarray) -> pd.DataFrame:
        """
        生成逐列报表：
        - 统计差异、形态差异、双峰状态、尾部/异常率指标
        - FC 的偏度/峰度、正态惩罚、严格双峰标记、正态硬违规标记
        """
        X_sub = self.X_all[best_idx, :]

        sub_q1, sub_med, sub_q3, sub_std = self._compute_stats(X_sub)
        sub_shape_all = np.nanpercentile(X_sub, self.shape_q, axis=0)

        shape_dist = np.mean(
            np.abs(sub_shape_all - self.ref_shape_all) / (self.scale_all + 1e-6),
            axis=0,
        )
        shape_dist = np.nan_to_num(shape_dist, nan=1e3, posinf=1e3, neginf=1e3)

        # 常规双峰
        sub_bimodal = np.array(
            [self._is_bimodal_cfg(X_sub[:, j]) for j in self.all_pos],
            dtype=bool,
        )

        # FC 严格双峰（仅 FC 列计算）
        sub_fc_bimodal_strict = np.full(len(self.all_cols), False, dtype=bool)
        for p in self.fc_pos_all:
            sub_fc_bimodal_strict[p] = self._is_bimodal_fc(X_sub[:, p])

        # 统计差异
        med_diff = np.abs(sub_med - self.ref_med_all)
        q1_diff = np.abs(sub_q1 - self.ref_q1_all)
        q3_diff = np.abs(sub_q3 - self.ref_q3_all)
        std_diff = np.abs(sub_std - self.ref_std_all)

        med_diff = np.nan_to_num(med_diff, nan=1e3, posinf=1e3, neginf=1e3)
        q1_diff = np.nan_to_num(q1_diff, nan=1e3, posinf=1e3, neginf=1e3)
        q3_diff = np.nan_to_num(q3_diff, nan=1e3, posinf=1e3, neginf=1e3)
        std_diff = np.nan_to_num(std_diff, nan=1e3, posinf=1e3, neginf=1e3)

        # 全 93 列尾部/异常率
        ref_p01_all = np.nanpercentile(self.X_ref_all, 1, axis=0)
        ref_p99_all = np.nanpercentile(self.X_ref_all, 99, axis=0)
        sub_p01_all = np.nanpercentile(X_sub, 1, axis=0)
        sub_p99_all = np.nanpercentile(X_sub, 99, axis=0)

        tail_ref_all = ref_p99_all - ref_p01_all
        tail_sub_all = sub_p99_all - sub_p01_all

        iqr_all = self.ref_q3_all - self.ref_q1_all
        tail_scale_all = np.where(
            tail_ref_all > 1e-6,
            tail_ref_all,
            np.where(iqr_all > 1e-6, iqr_all, np.where(self.ref_std_all > 1e-6, self.ref_std_all, 1.0)),
        )
        tail_diff_norm = np.abs(tail_sub_all - tail_ref_all) / (tail_scale_all + 1e-6)

        iqr_safe_all = np.where(
            iqr_all > 1e-6,
            iqr_all,
            np.where(self.ref_std_all > 1e-6, self.ref_std_all, 1.0),
        )

        low15_all = self.ref_q1_all - 1.5 * iqr_safe_all
        up15_all = self.ref_q3_all + 1.5 * iqr_safe_all
        low30_all = self.ref_q1_all - 3.0 * iqr_safe_all
        up30_all = self.ref_q3_all + 3.0 * iqr_safe_all

        ref_out15 = np.nanmean((self.X_ref_all < low15_all) | (self.X_ref_all > up15_all), axis=0)
        sub_out15 = np.nanmean((X_sub < low15_all) | (X_sub > up15_all), axis=0)
        ref_out30 = np.nanmean((self.X_ref_all < low30_all) | (self.X_ref_all > up30_all), axis=0)
        sub_out30 = np.nanmean((X_sub < low30_all) | (X_sub > up30_all), axis=0)

        tail_diff_norm = np.nan_to_num(tail_diff_norm, nan=1e3, posinf=1e3, neginf=1e3)
        ref_out15 = np.nan_to_num(ref_out15, nan=0.0, posinf=0.0, neginf=0.0)
        sub_out15 = np.nan_to_num(sub_out15, nan=0.0, posinf=0.0, neginf=0.0)
        ref_out30 = np.nan_to_num(ref_out30, nan=0.0, posinf=0.0, neginf=0.0)
        sub_out30 = np.nan_to_num(sub_out30, nan=0.0, posinf=0.0, neginf=0.0)

        out_rate15_diff = np.abs(sub_out15 - ref_out15)
        out_rate30_excess = np.maximum(0.0, sub_out30 - (ref_out30 + 0.01))

        # 列类型标签
        is_fc = np.array([str(c).startswith("FC") for c in self.all_cols], dtype=bool)
        target_set = set(self.target_cols)
        is_target = np.array([c in target_set for c in self.all_cols], dtype=bool)

        # 参考双峰特殊规则映射（仅 target 列）
        target_bi_map = {c: bool(b) for c, b in zip(self.target_cols, self.t_bimodal)}
        special_rule = np.array([target_bi_map.get(c, False) for c in self.all_cols], dtype=bool)

        # FC 偏度/峰度与正态惩罚
        fc_skew = np.full(len(self.all_cols), np.nan, dtype=float)
        fc_kurt = np.full(len(self.all_cols), np.nan, dtype=float)

        for p in self.fc_pos_all:
            y = X_sub[:, p]
            y = y[np.isfinite(y)]
            if y.size >= 8:
                fc_skew[p] = float(skew(y, bias=False))
                fc_kurt[p] = float(kurtosis(y, fisher=False, bias=False))

        sk_thr = self.fc_normal_skew_thr if self.prioritize_fc else self.normal_skew_thr
        ku_thr = self.fc_normal_kurt_thr if self.prioritize_fc else self.normal_kurt_thr

        fc_norm_pen = np.full(len(self.all_cols), np.nan, dtype=float)
        fc_normality_gap_strict = np.full(len(self.all_cols), np.nan, dtype=float)
        fc_normality_hard_violation = np.full(len(self.all_cols), False, dtype=bool)

        for p in self.fc_pos_all:
            skv = fc_skew[p]
            kuv = fc_kurt[p]
            if np.isfinite(skv) and np.isfinite(kuv):
                gap = max(0.0, abs(skv) - sk_thr) + max(0.0, abs(kuv - 3.0) - ku_thr)
                fc_normality_gap_strict[p] = gap
                fc_norm_pen[p] = gap * (self.fc_norm_weight_mult if self.prioritize_fc else 1.0)

                if self.prioritize_fc:
                    fc_normality_hard_violation[p] = (
                        (abs(skv) > (sk_thr + self.fc_norm_hard_margin))
                        or (abs(kuv - 3.0) > (ku_thr + self.fc_norm_hard_margin))
                    )

        max_diff = np.maximum.reduce([med_diff, q1_diff, q3_diff, std_diff])
        above_thr = max_diff > self.stat_threshold

        report = pd.DataFrame(
            {
                "column": self.all_cols,
                "is_target": is_target,
                "is_fc": is_fc,
                "ref_bimodal": self.ref_bimodal_all,
                "target_special_rule_ref_bimodal": special_rule,
                "subset_bimodal": sub_bimodal,
                "subset_fc_bimodal_strict": sub_fc_bimodal_strict,
                "shape_distance": shape_dist,
                "median_diff": med_diff,
                "q1_diff": q1_diff,
                "q3_diff": q3_diff,
                "std_diff": std_diff,
                "max_diff": max_diff,
                "above_stat_threshold": above_thr,
                "tail_diff_norm": tail_diff_norm,
                "out_rate15_ref": ref_out15,
                "out_rate15_sub": sub_out15,
                "out_rate15_diff": out_rate15_diff,
                "out_rate30_ref": ref_out30,
                "out_rate30_sub": sub_out30,
                "out_rate30_excess": out_rate30_excess,
                "fc_skew": fc_skew,
                "fc_kurtosis": fc_kurt,
                "fc_normality_penalty": fc_norm_pen,
                "fc_normality_gap_strict": fc_normality_gap_strict,
                "fc_normality_hard_violation": fc_normality_hard_violation,
            }
        )

        return report

    def fit(self, df_C: pd.DataFrame, df_ref: pd.DataFrame):
        """
        多起点搜索主入口。
        返回：
        - subset_df: 最优子集
        - summary: 汇总指标
        - col_report: 逐列报表
        - history_df: 迭代历史
        """
        self._prepare(df_C=df_C, df_ref=df_ref)

        best_global_idx = None
        best_global_eval = None
        history_rows = []

        for run in range(1, self.n_starts + 1):
            init_idx = self._sample_initial_indices()
            best_idx, best_eval, run_hist = self._optimize_once(init_idx)

            for row in run_hist:
                row["run"] = run
                history_rows.append(row)

            if (best_global_eval is None) or (best_eval["score"] < best_global_eval["score"]):
                best_global_idx = best_idx
                best_global_eval = best_eval

        if best_global_idx is None:
            raise RuntimeError("搜索失败：未得到有效解。")

        final_eval = self._evaluate(best_global_idx, detail=True)

        # 恢复输出顺序
        subset_df = self.df_c.iloc[best_global_idx].copy()
        subset_df = subset_df.sort_values("__orig_index__")
        subset_df.index = subset_df["__orig_index__"].to_numpy()
        subset_df = subset_df.drop(columns=["__orig_index__"])

        col_report = self._build_column_report(best_global_idx)
        history_df = pd.DataFrame(history_rows)

        bp, pp, mp = final_eval["judge_props"]

        summary = {
            "target_mode": self.target_mode,
            "prioritize_fc": bool(self.prioritize_fc),
            "n_target_columns": len(self.target_cols),
            "n_fc_columns": len(self.fc_cols),
            "subset_size": int(len(best_global_idx)),
            "size_ok": bool(final_eval["size_ok"]),
            "judge_ok": bool(final_eval["judge_ok"]),
            "judge_best": float(bp),
            "judge_pass": float(pp),
            "judge_medium": float(mp),
            "all_non_bimodal": bool(final_eval["bimodal_count"] == 0),
            "bimodal_columns_count": int(final_eval["bimodal_count"]),
            "fc_bimodal_strict_count": int(final_eval["fc_bimodal_count"]),
            "fc_norm_hard_count": int(final_eval["fc_norm_hard_count"]),
            "extreme_hard_count": int(final_eval["extreme_hard_count"]),
            "stat_violation_count": int(final_eval["stat_violation_count"]),
            "avg_stat_diff": float(final_eval["avg_stat_diff"]),
            "shape_penalty": float(final_eval["shape_penalty"]),
            "tail_penalty": float(final_eval["tail_penalty"]),
            "outlier_penalty": float(final_eval["outlier_penalty"]),
            "extreme_penalty": float(final_eval["extreme_penalty"]),
            "fc_normality_penalty": float(final_eval["normality_penalty"]),
            "final_score": float(final_eval["score"]),
            "fixed_sample_size": int(self.fixed_n),
            "fixed_counts": {
                "BEST": int(self.fixed_counts[0]),
                "PASS": int(self.fixed_counts[1]),
                "MEDIUM": int(self.fixed_counts[2]),
            },
        }

        return subset_df, summary, col_report, history_df


# =========================
# 可直接运行示例
# =========================
if __name__ == "__main__":
    # 1) 输入路径
    path_c = r"C:\Users\HP\Desktop\code\C.csv"
    path_c30 = r"C:\Users\HP\Desktop\code\C30.csv"
    ref_group = "0.0XM C3.0_CGS"

    # 2) 输出路径
    out_subset = r"C:\Users\HP\Desktop\code\optimal_subset.xlsx"
    out_col_report = r"C:\Users\HP\Desktop\code\col_report.xlsx"
    out_history = r"C:\Users\HP\Desktop\code\search_history.xlsx"
    out_summary = r"C:\Users\HP\Desktop\code\summary.xlsx"
    out_counts = r"C:\Users\HP\Desktop\code\group_counts.xlsx"

    # 3) 读数据
    df_C = load_table(path_c)
    df_C30 = load_table(path_c30)

    # 4) 按 # 分组计数，取 count > 320 的第一个分组作为候选池
    hash_norm = df_C["#"].fillna("<NA>").astype(str).str.strip()
    counts = hash_norm.value_counts()
    counts_df = (
        counts.rename_axis("#")
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    counts_df.to_excel(out_counts, index=False)

    classes_over_320 = counts[counts > 320].index.tolist()
    if not classes_over_320:
        raise ValueError("未找到样本量 > 320 的分组，请检查 df_C['#']。")

    first_val = classes_over_320[0]
    C_data = df_C[hash_norm == first_val].copy()
    print(f"Selected first class: {first_val}, n={len(C_data)}")

    # 5) 参考组
    data_ref = df_C30[df_C30["#"] == ref_group].copy()
    if data_ref.empty:
        raise ValueError("参考组为空，请检查 df_C30 的过滤条件。")

    # 6) 是否更看重 FC 列（你的可选输入）
    use_fc_priority = True

    # 7) 初始化优化器
    optimizer = FastSubsetOptimizer(
        target_mode="all_93",
        stat_threshold=3.0,
        size_min=320,
        size_max=340,
        target_size=330,
        judge_tolerance=0.015,
        n_starts=5,
        max_iter=900,
        early_stop=220,
        random_seed=42,
        # 常规软项权重
        w_shape=1.2,
        w_stat=1.0,
        w_norm=0.6,
        w_stat_violation=0.2,
        w_tail=1.2,
        w_outlier=2.5,
        w_extreme=8.0,
        hard_extreme_margin=0.02,
        # 常规双峰参数
        bimodal_sep_ratio=0.08,
        bimodal_valley_ratio=0.78,
        bimodal_min_side_mass=0.12,
        # FC 优先模式参数
        prioritize_fc=use_fc_priority,
        fc_weight_mult=2.5,
        fc_bimodal_peak_height_ratio=0.06,
        fc_bimodal_sep_ratio=0.05,
        fc_bimodal_valley_ratio=0.92,
        fc_bimodal_min_side_mass=0.08,
        fc_normal_skew_thr=0.30,
        fc_normal_kurt_thr=0.80,
        fc_norm_weight_mult=2.5,
        fc_norm_hard_margin=0.20,
    )

    # 8) 运行优化
    subset_df, summary, col_report, history_df = optimizer.fit(df_C=C_data, df_ref=data_ref)

    # 9) 输出结果
    print("\nSummary:")
    for k, v in summary.items():
        print(f"  {k}: {v}")

    subset_df.to_excel(out_subset, index=False)
    col_report.to_excel(out_col_report, index=False)
    history_df.to_excel(out_history, index=False)
    pd.DataFrame([summary]).to_excel(out_summary, index=False)

    print("\nSaved:")
    print(f"  subset -> {out_subset}")
    print(f"  col_report -> {out_col_report}")
    print(f"  history -> {out_history}")
    print(f"  summary -> {out_summary}")
    print(f"  group_counts -> {out_counts}")

Selected first class: 5C-C5, n=1727

Summary:
  target_mode: all_93
  prioritize_fc: True
  n_target_columns: 93
  n_fc_columns: 22
  subset_size: 330
  size_ok: True
  judge_ok: True
  judge_best: 0.47575757575757577
  judge_pass: 0.44545454545454544
  judge_medium: 0.07878787878787878
  all_non_bimodal: True
  bimodal_columns_count: 0
  fc_bimodal_strict_count: 0
  fc_norm_hard_count: 2
  extreme_hard_count: 4
  stat_violation_count: 8
  avg_stat_diff: 0.937287840164139
  shape_penalty: 0.5778999258812318
  tail_penalty: 0.22542942097750623
  outlier_penalty: 0.04591964521205103
  extreme_penalty: 0.003669880511120296
  fc_normality_penalty: 0.24847187101840973
  final_score: 1400002.2072227485
  fixed_sample_size: 330
  fixed_counts: {'BEST': 157, 'PASS': 147, 'MEDIUM': 26}

Saved:
  subset -> C:\Users\HP\Desktop\code\optimal_subset.xlsx
  col_report -> C:\Users\HP\Desktop\code\col_report.xlsx
  history -> C:\Users\HP\Desktop\code\search_history.xlsx
  summary -> C:\Users\HP\Desktop

In [25]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

# 如果你还没定义 data_ref93，可以取消下面一行注释
data_ref93 = df_C30[df_C30["#"] == "0.0XM C3.0_CGS"].copy()

# 取从 EFL 开始的 93 列
start_idx = data_ref93.columns.get_loc("EFL")
cols_93 = list(data_ref93.columns[start_idx:start_idx + 93])

# 画图风格
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
})

def to_numeric_series(df, col):
    return pd.to_numeric(df[col], errors="coerce").dropna()

def common_ylim(s1, s2, pad_ratio=0.03):
    combined = pd.concat([s1, s2], ignore_index=True).dropna()
    if combined.empty:
        return (-1, 1)

    lo = float(combined.min())   # 真实最小值
    hi = float(combined.max())   # 真实最大值

    if not np.isfinite(lo) or not np.isfinite(hi):
        return (-1, 1)

    if hi <= lo:
        pad = 1.0 if lo == 0 else abs(lo) * 0.05
        return (lo - pad, hi + pad)

    pad = (hi - lo) * pad_ratio
    return (lo - pad, hi + pad)

def draw_violin_box(ax, series, color):
    sns.violinplot(
        y=series,
        ax=ax,
        inner=None,
        cut=0,
        color=color,
        linewidth=0,
        saturation=0.7,
        width=0.65,
        zorder=1,
    )

    # 小提琴填充再淡一点
    for artist in ax.collections:
        artist.set_alpha(0.25)
        artist.set_edgecolor("none")

    bp = ax.boxplot(
        series.dropna(),
        vert=True,
        widths=0.22,
        showfliers=True,   # 打开离群点
        whis=1.5,
        patch_artist=True,
        zorder=3,
    )
    plt.setp(
        bp["fliers"],
        marker="o",
        markersize=2.2,
        alpha=0.35,
        markerfacecolor="black",
        markeredgecolor="none",
    )

    plt.setp(bp["boxes"], facecolor="none", edgecolor="black", linewidth=1.2, zorder=3)
    plt.setp(bp["whiskers"], color="black", linewidth=1.0, zorder=3)
    plt.setp(bp["caps"], color="black", linewidth=1.0, zorder=3)
    plt.setp(bp["medians"], color="black", linewidth=1.3, zorder=3)

    ax.grid(axis="y", alpha=0.25, linestyle="--", linewidth=0.6)
    ax.set_xlabel("")
    ax.tick_params(axis="x", bottom=False, labelbottom=False)
    sns.despine(ax=ax, left=False, bottom=True)

pdf_path = r"C:\Users\HP\Desktop\code\optimal_subset_vs_data_ref93.pdf"

# 每页画多少行，太多会太挤，太少会页数很多
rows_per_page = 1

with PdfPages(pdf_path) as pdf:
    for page_start in range(0, len(cols_93), rows_per_page):
        page_cols = cols_93[page_start:page_start + rows_per_page]
        n_rows = len(page_cols)

        fig, axes = plt.subplots(
            nrows=n_rows,
            ncols=2,
            figsize=(14, 6.5 * n_rows),
            constrained_layout=True
        )

        if n_rows == 1:
            axes = np.array([axes])

        fig.suptitle(
            f"Optimal Subset vs Reference Distribution  |  Columns {page_start + 1} - {page_start + n_rows}",
            fontsize=12,
            fontweight="bold",
            y=1.01
        )

        for i, col in enumerate(page_cols):
            ref_s = to_numeric_series(data_ref93, col)
            sub_s = to_numeric_series(subset_df, col)
            ylim = common_ylim(ref_s, sub_s)

            # 左边：目标分布
            ax_l = axes[i, 0]
            draw_violin_box(ax_l, ref_s, color="#4C78A8")
            ax_l.set_ylim(ylim)
            ax_l.set_title("Target", loc="center", pad=4)
            ax_l.set_ylabel("")
            ax_l.text(
                -0.18, 0.5, col,
                transform=ax_l.transAxes,
                rotation=90,
                va="center",
                ha="center",
                fontsize=9,
                fontweight="bold"
            )

            # 右边：找到的数据
            ax_r = axes[i, 1]
            draw_violin_box(ax_r, sub_s, color="#F58518")
            ax_r.set_ylim(ylim)
            ax_r.set_title("Selected", loc="center", pad=4)
            ax_r.set_ylabel("")

            # 如果某列两边都没数据，给一个提示
            if ref_s.empty and sub_s.empty:
                for ax in (ax_l, ax_r):
                    ax.text(
                        0.5, 0.5, "No data",
                        transform=ax.transAxes,
                        ha="center",
                        va="center",
                        fontsize=10,
                        color="gray"
                    )

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Saved to: {pdf_path}")

Saved to: C:\Users\HP\Desktop\code\optimal_subset_vs_data_ref93.pdf
